[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prabudhd2003/Calm-to-Conflict/blob/main/MulT.ipynb)

# Calm to Conflict — Dyadic Multimodal Transformer

End-to-end pipeline for utterance-level conflict descent prediction in dyadic video interactions. Three model sections run in sequence:

1. **Dyadic MulT Ablation** — 4-stream (Text + Audio + Self-Video + Partner-Video) modality ablation
2. **Visual Feature Ablation** — fine-grained ablation within the 92-dim visual stream
3. **Optimized Audiovisual Fusion** — targeted Audio + FAU models (best final configuration)

Each section is self-contained: run its cells top-to-bottom. All shared utilities (imports, seeds, paths, shared model primitives) are defined once in **Section 0** below.

---
# Section 0 — Shared Configuration

Imports, output paths, random seeds, and model primitives shared across all sections. **Run this section first.**

## Imports

In [1]:
import os
import json
import random
import copy
import math
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)
from sklearn.model_selection import GroupShuffleSplit
from tqdm import tqdm

import matplotlib.pyplot as plt
import seaborn as sns


## Output Paths

All file I/O goes through `PATHS`. Change these once to redirect all output.

In [2]:
# Output directory layout

PATHS = {
    "csv":     "out/master_dataset_dyadic.csv",
    "models":  "out/models",
    "plots":   "out/plots",
    "results": "out/results",
}

# Embedding file paths
EMBEDDINGS = {
    "text":            "embeddings/text_sequences_v1.pt",
    "audio":           "embeddings/audio_sequences_v2.pt",
    "video_self":      "embeddings/video_self_sequences_v1.pt",
    "video_partner":   "embeddings/video_partner_sequences_v1.pt",
    "video_feat_dims": "embeddings/video_feature_dims.json",
}

for path in PATHS.values():
    if not path.endswith(".csv"):
        os.makedirs(path, exist_ok=True)


## Random Seeds


In [3]:
# Multi-seed evaluation
RANDOM_SEEDS = [42, 68, 92, 105, 208]

## Shared Utilities

`set_seed`, `group_split`, and the JSON results helper used by all sections.

In [4]:
def set_seed(seed: int = 42) -> None:
    """Set all relevant random states for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def group_split(df: pd.DataFrame, test_size: float = 0.20, seed: int = 42):
    """
    Group-based train/test split that keeps all utterances from the same
    speaker file in the same partition, preventing cross-conversation leakage.
    """
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    groups = df["file_id"].values
    train_idx, test_idx = next(gss.split(df, groups=groups))
    return df.iloc[train_idx].reset_index(drop=True), df.iloc[test_idx].reset_index(drop=True)


def load_embedding(key: str):
    """Load a .pt embedding file, unwrapping dict wrappers when present."""
    raw = torch.load(EMBEDDINGS[key], map_location="cpu", weights_only=False)
    return raw.get("audio_sequences", raw) if isinstance(raw, dict) else raw


def save_results_json(path: str, new_entries: dict) -> None:
    """Merge new_entries into an existing JSON results file (or create it)."""
    if os.path.exists(path):
        with open(path) as f:
            existing = json.load(f)
        existing.update(new_entries)
        new_entries = existing
    with open(path, "w") as f:
        json.dump(new_entries, f, indent=2)


def aggregate_seeds(acc_list: list, f1_list: list) -> dict:
    """Return mean/std summary dict from per-seed acc and f1 lists."""
    acc_arr, f1_arr = np.array(acc_list), np.array(f1_list)
    return {
        "acc_mean":     float(acc_arr.mean()),
        "acc_std":      float(acc_arr.std()),
        "f1_mean":      float(f1_arr.mean()),
        "f1_std":       float(f1_arr.std()),
        "acc_per_seed": [float(x) for x in acc_arr],
        "f1_per_seed":  [float(x) for x in f1_arr],
        "seeds":        list(RANDOM_SEEDS),
    }


def print_summary_table(title: str, summary: dict) -> None:
    """Print a formatted mean ± std results table."""
    print("\n" + "="*75)
    print(f"{title}  (seeds: {RANDOM_SEEDS})")
    print("="*75)
    print(f"{'Experiment':<35} {'Acc (%)':<22} {'F1 (macro)'}")
    print("-"*75)
    for name, r in summary.items():
        print(f"{name:<35} {r['acc_mean']*100:.2f} ± {r['acc_std']*100:.2f}%{'':<8}"
              f"{r['f1_mean']:.4f} ± {r['f1_std']:.4f}")
    print("="*75)


## Shared Model Primitives

`PositionalEncoding`, `AttentionPool`, `CrossModalAttentionBlock`, and the cosine LR schedule — defined once and reused by all three model sections.

In [5]:
# ════════════════════════════════════════════════════════════════════════════
#  Shared Building Blocks
#  Used by all three model sections below.
#  PositionalEncoding, AttentionPool, CrossModalAttentionBlock,
#  get_cosine_schedule_with_warmup
# ════════════════════════════════════════════════════════════════════════════

class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding injected after input projection."""
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 2000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, 1, d_model)
        pe[:, 0, 0::2] = torch.sin(position * div_term)
        pe[:, 0, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe)

    def forward(self, x):
        x = x.transpose(0, 1)
        x = x + self.pe[: x.size(0)]
        return self.dropout(x.transpose(0, 1))


class AttentionPool(nn.Module):
    """Collapses a temporal sequence [B, T, D] to [B, D] via learned attention weights."""
    def __init__(self, dim: int):
        super().__init__()
        self.score = nn.Linear(dim, 1)

    def forward(self, x):
        w = torch.softmax(self.score(x), dim=1)
        return (x * w).sum(dim=1)


class CrossModalAttentionBlock(nn.Module):
    """
    Single cross-modal attention sub-layer:  query ← (key, value)
      Attention → residual + LayerNorm → FFN → residual + LayerNorm
    """
    def __init__(self, embed_dim: int, num_heads: int, dropout: float = 0.2):
        super().__init__()
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=embed_dim, num_heads=num_heads,
            dropout=dropout, batch_first=True
        )
        self.norm1   = nn.LayerNorm(embed_dim)
        self.norm2   = nn.LayerNorm(embed_dim)
        self.ffn     = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * 2, embed_dim),
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, query_seq, key_value_seq):
        attn_out, _ = self.cross_attn(
            query=query_seq, key=key_value_seq, value=key_value_seq
        )
        x = self.norm1(query_seq + self.dropout(attn_out))
        x = self.norm2(x + self.dropout(self.ffn(x)))
        return x


def get_cosine_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps):
    """Cosine annealing LR schedule with a linear warm-up phase."""
    def lr_lambda(step):
        if step < num_warmup_steps:
            return float(step) / float(max(1, num_warmup_steps))
        progress = float(step - num_warmup_steps) / float(
            max(1, num_training_steps - num_warmup_steps)
        )
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


## Shared Plot Helper

In [6]:
def save_training_plots(
    experiment_name: str,
    train_losses:    list,
    val_losses:      list,
    val_accuracies:  list,
    best_epoch:      int,
    final_preds:     list,
    final_targets:   list,
    plot_dir:        str = None,
) -> None:
    """Save loss/accuracy curves and confusion matrix for one training run."""
    plot_dir = plot_dir or PATHS["plots"]
    os.makedirs(plot_dir, exist_ok=True)
    epochs_run = len(train_losses)

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(range(1, epochs_run + 1), train_losses,
             label="Train Loss", color="steelblue", linewidth=2)
    plt.plot(range(1, epochs_run + 1), val_losses,
             label="Val Loss", color="tomato", linewidth=2, linestyle="--")
    plt.axvline(x=best_epoch, color="green", linestyle=":", label=f"Best Epoch {best_epoch}")
    plt.title(f"Loss: {experiment_name}", fontsize=12)
    plt.xlabel("Epochs"); plt.ylabel("Loss")
    plt.legend(); plt.grid(True, alpha=0.3)

    plt.subplot(1, 2, 2)
    plt.plot(range(1, epochs_run + 1), [a * 100 for a in val_accuracies],
             label="Val Accuracy", color="mediumseagreen", linewidth=2)
    plt.axvline(x=best_epoch, color="red", linestyle=":", label=f"Best Epoch {best_epoch}")
    plt.title(f"Accuracy: {experiment_name}", fontsize=12)
    plt.xlabel("Epochs"); plt.ylabel("Accuracy (%)")
    plt.legend(); plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(plot_dir, f"{experiment_name}_curves.png"), dpi=200)
    try:
        plt.show(block=False); plt.pause(2)
    except Exception:
        pass
    plt.close()

    cm = confusion_matrix(final_targets, final_preds)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Predicted Calm", "Predicted Conflict"],
                yticklabels=["Actual Calm",    "Actual Conflict"],
                annot_kws={"size": 14})
    plt.title(f"Confusion Matrix: {experiment_name}", fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(plot_dir, f"{experiment_name}_cm.png"), dpi=200)
    try:
        plt.show(block=False); plt.pause(2)
    except Exception:
        pass
    plt.close()
    print(f"Plots saved → {plot_dir}/")

---
# Section 1 — Video Feature Extraction

Extracts and saves the 92-dim per-frame visual sequences (FAU + head + gaze + body) from raw `.npz` files for every speaker in the dataset. Produces `video_self_sequences_v1.pt`, `video_partner_sequences_v1.pt`, and `video_feature_dims.json`. **Only needs to be run once.**

## Configuration

In [ ]:
MASTER_CSV   = "out/master_dataset_dyadic.csv"
OUTPUT_SELF  = "embeddings/video_self_sequences_v1.pt"
OUTPUT_PART  = "embeddings/video_partner_sequences_v1.pt"
OUTPUT_DIMS  = "embeddings/video_feature_dims.json"

ORIGINAL_FPS = 30
TARGET_FPS   = 5
NUM_WORKERS  = 4

## NPZ Feature Loader

Reads FAU, head orientation, gaze, and body pose arrays from a single `.npz` file and stacks them into a single `[T, D_total]` matrix.

In [ ]:
def _load_npz_features(npz_path: str):
    """
    Loads FAU + head + gaze + body from an NPZ file.

    Returns
    -------
    matrix      : np.ndarray [num_frames, D_total]
    dim_info    : dict with fau_dim, head_dim, gaze_dim, body_dim
                  (None if load fails)
    """
    try:
        d = np.load(npz_path, allow_pickle=True)

        if   'movement_v4:FAUValue' in d.files: au = d['movement_v4:FAUValue']
        elif 'movement:FAUValue'    in d.files: au = d['movement:FAUValue']
        else: return None, None

        n = au.shape[0]

        if   'movement_v4:head_encodings' in d.files: head = d['movement_v4:head_encodings']
        elif 'movement:head_encodings'    in d.files: head = d['movement:head_encodings']
        else: head = np.zeros((n, 6))

        if   'movement_v4:gaze_encodings' in d.files: gaze = d['movement_v4:gaze_encodings']
        elif 'movement:gaze_encodings'    in d.files: gaze = d['movement:gaze_encodings']
        else: gaze = np.zeros((n, 6))

        if   'smplh:body_pose' in d.files: body = d['smplh:body_pose']
        else: body = np.zeros((n, 63))

        # Flatten [Frames, Joints, 3] → [Frames, Joints*3]
        au   = au.reshape(n, -1)
        head = head.reshape(n, -1)
        gaze = gaze.reshape(n, -1)
        body = body.reshape(n, -1)

        min_f = min(au.shape[0], head.shape[0], gaze.shape[0], body.shape[0])
        matrix = np.column_stack([
            au[:min_f], head[:min_f], gaze[:min_f], body[:min_f]
        ])

        dim_info = {
            "fau_dim":  au.shape[1],
            "head_dim": head.shape[1],
            "gaze_dim": gaze.shape[1],
            "body_dim": body.shape[1],
        }
        return matrix, dim_info

    except Exception:
        return None, None

## Per-File Processor

Processes one speaker file: temporally downsamples, pads/truncates to a fixed length, and extracts both the speaker clip and the 2-second partner reaction window.

In [ ]:
def process_single_file(args):
    self_npz, partner_npz, group_df, max_video_len = args
    self_results    = {}
    partner_results = {}

    self_matrix, dim_info = _load_npz_features(self_npz)
    if self_matrix is None:
        return {}, {}, None

    if partner_npz and os.path.exists(partner_npz):
        partner_matrix, _ = _load_npz_features(partner_npz)
        if partner_matrix is None:
            partner_matrix = np.zeros_like(self_matrix)
        min_len        = min(self_matrix.shape[0], partner_matrix.shape[0])
        self_matrix    = self_matrix[:min_len]
        partner_matrix = partner_matrix[:min_len]
    else:
        partner_matrix = np.zeros_like(self_matrix)

    step      = ORIGINAL_FPS // TARGET_FPS   # 6
    offset_fr = int(2.0 * ORIGINAL_FPS)      # 60 frames = 2 sec reaction window

    def process_clip(clip):
        if step > 0 and len(clip) > step:
            clip = clip[::step]
        t = torch.tensor(clip, dtype=torch.float32)
        cur = t.shape[0]
        if cur > max_video_len:
            t = t[:max_video_len]
        elif cur < max_video_len:
            t = F.pad(t, (0, 0, 0, max_video_len - cur), "constant", 0)
        return t

    for _, row in group_df.iterrows():
        sid      = row['sample_id']
        start_fr = int(row['start_sec'] * ORIGINAL_FPS)
        end_fr   = min(int(row['end_sec'] * ORIGINAL_FPS), self_matrix.shape[0])

        self_results[sid]    = process_clip(self_matrix[start_fr:end_fr])
        partner_end          = min(end_fr + offset_fr, partner_matrix.shape[0])
        partner_results[sid] = process_clip(partner_matrix[end_fr:partner_end])

    return self_results, partner_results, dim_info

## Extraction Runner

Parallelised extraction over all speaker files with checkpoint resumption. Saves `.pt` embedding dicts and the `video_feature_dims.json` boundary metadata.

In [ ]:
def extract_dyadic_video_sequences():
    df = pd.read_csv(MASTER_CSV)
    p95_duration  = df['duration_sec'].quantile(0.95)
    MAX_VIDEO_LEN = math.ceil(p95_duration * TARGET_FPS)
    print(f"95th percentile duration : {p95_duration:.2f}s")
    print(f"MAX_VIDEO_LEN            : {MAX_VIDEO_LEN} frames @ {TARGET_FPS} fps")

    # ---- Checkpoint resume ----
    if os.path.exists(OUTPUT_SELF) and os.path.exists(OUTPUT_PART):
        self_dict    = torch.load(OUTPUT_SELF, weights_only=False)
        partner_dict = torch.load(OUTPUT_PART, weights_only=False)
        done         = set(self_dict.keys())
        df_todo      = df[~df['sample_id'].isin(done)]
        print(f"Resuming — {len(done)} done, {len(df_todo)} remaining.")
    else:
        self_dict    = {}
        partner_dict = {}
        df_todo      = df

    global_dim_info = None   # will be set from the first successful file

    if len(df_todo) > 0:
        tasks = []
        for self_npz, group in df_todo.groupby('npz_path'):
            partner_npz = group['partner_npz_path'].iloc[0]
            if pd.isna(partner_npz) or partner_npz == "":
                partner_npz = ""
            tasks.append((self_npz, partner_npz, group, MAX_VIDEO_LEN))

        print(f"\nGrouped into {len(tasks)} unique speaker files.")
        print(f"Starting multi-core extraction ({NUM_WORKERS} workers)...")

        save_interval = 50
        with ProcessPoolExecutor(max_workers=NUM_WORKERS) as executor:
            futures = {executor.submit(process_single_file, t): t for t in tasks}
            for i, future in enumerate(tqdm(as_completed(futures),
                                            total=len(futures),
                                            desc="Processing files")):
                try:
                    s_res, p_res, dim_info = future.result()
                    self_dict.update(s_res)
                    partner_dict.update(p_res)
                    if dim_info is not None and global_dim_info is None:
                        global_dim_info = dim_info
                except Exception as e:
                    print(f"  Worker error: {e}")

                if (i + 1) % save_interval == 0:
                    torch.save(self_dict,    OUTPUT_SELF)
                    torch.save(partner_dict, OUTPUT_PART)

        torch.save(self_dict,    OUTPUT_SELF)
        torch.save(partner_dict, OUTPUT_PART)
    else:
        print("Embeddings already complete — generating metadata only.")

    # ---- Fallback: if global_dim_info was never set (all workers failed,
    #      or df_todo was empty), scan the CSV for any readable NPZ file. ----
    if global_dim_info is None:
        print("Discovering feature dims from CSV NPZ paths...")
        for npz_path in df['npz_path'].dropna().unique():
            _, info = _load_npz_features(npz_path)
            if info is not None:
                global_dim_info = info
                print(f"  Dims read from: {npz_path}")
                break
        if global_dim_info is None:
            print("ERROR: Could not read feature dims from any NPZ file in the CSV.")
            print("       Check that npz_path column in the CSV points to accessible files.")

    # ---- Save feature boundary metadata ----
    if global_dim_info is not None:
        fau_end  = global_dim_info['fau_dim']
        head_end = fau_end  + global_dim_info['head_dim']
        gaze_end = head_end + global_dim_info['gaze_dim']
        body_end = gaze_end + global_dim_info['body_dim']

        meta = {
            **global_dim_info,
            "total_dim":  body_end,
            "fau_slice":  [0,        fau_end],
            "head_slice": [fau_end,  head_end],
            "gaze_slice": [head_end, gaze_end],
            "body_slice": [gaze_end, body_end],
        }
        with open(OUTPUT_DIMS, "w") as f:
            json.dump(meta, f, indent=2)
        print(f"\nFeature boundary metadata → {OUTPUT_DIMS}")
        print(f"  FAU : dim={meta['fau_dim']:>3}  slice [{meta['fau_slice'][0]:>2}:{meta['fau_slice'][1]:>3}]")
        print(f"  Head: dim={meta['head_dim']:>3}  slice [{meta['head_slice'][0]:>2}:{meta['head_slice'][1]:>3}]")
        print(f"  Gaze: dim={meta['gaze_dim']:>3}  slice [{meta['gaze_slice'][0]:>2}:{meta['gaze_slice'][1]:>3}]")
        print(f"  Body: dim={meta['body_dim']:>3}  slice [{meta['body_slice'][0]:>2}:{meta['body_slice'][1]:>3}]")

    print(f"\nSaved self    embeddings → {OUTPUT_SELF}")
    print(f"Saved partner embeddings → {OUTPUT_PART}")
    print(f"Total samples in self    : {len(self_dict)}")
    print(f"Total samples in partner : {len(partner_dict)}")

    n_zero = sum(
        1 for v in partner_dict.values() if v.abs().sum().item() == 0
    )
    print(f"Partner samples all-zeros (no partner): {n_zero}")


if __name__ == "__main__":
    extract_dyadic_video_sequences()

---
# Section 2 — Dyadic MulT: Modality Ablation

Full 4-stream Multimodal Transformer (Text + Audio + Self-Video + Partner-Video). Runs 9 ablation experiments × `RANDOM_SEEDS` to isolate the contribution of each modality. Saves one best-seed `.pt` per experiment and a consolidated `dyadic_mult_results.json`.

## Dataset & MixUp

In [ ]:
class DyadicSequenceDataset(Dataset):
    """
    Returns (text, audio, video_self, video_partner, label) tuples.

    If a partner embedding is missing or all-zeros (57 samples),
    the partner tensor is kept as zeros so the model can learn to
    treat it as a null signal — consistent with training-time behaviour.
    """

    def __init__(self, df, text_dict, audio_dict, self_video_dict, partner_video_dict):
        self.samples = []
        missing = 0
        for _, row in df.iterrows():
            s_id = row["sample_id"]
            if (s_id in text_dict and s_id in audio_dict
                    and s_id in self_video_dict):
                # Partner may be missing for 57 samples; fall back to zeros
                partner_v = partner_video_dict.get(
                    s_id,
                    torch.zeros_like(self_video_dict[s_id])
                )
                self.samples.append({
                    "label":          int(row["label"]),
                    "text":           text_dict[s_id],
                    "audio":          audio_dict[s_id],
                    "video_self":     self_video_dict[s_id],
                    "video_partner":  partner_v,
                })
            else:
                missing += 1
        if missing:
            print(f"  [Dataset] Skipped {missing} samples missing from at least one embedding dict.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        return (
            item["text"].clone().float(),
            item["audio"].clone().float(),
            item["video_self"].clone().float(),
            item["video_partner"].clone().float(),
            torch.tensor(item["label"], dtype=torch.long),
        )

# MIXUP (extended to 4 streams)
def mixup_batch(x_text, x_audio, x_vs, x_vp, y, alpha: float = 0.2):
    """In-batch MixUp across all 4 input streams."""
    batch_size  = y.size(0)
    num_classes = 2

    if alpha <= 0:
        y_soft = torch.zeros(batch_size, num_classes, device=y.device)
        y_soft.scatter_(1, y.unsqueeze(1), 1.0)
        return x_text, x_audio, x_vs, x_vp, y_soft

    lam = np.random.beta(alpha, alpha)
    lam = max(lam, 1 - lam)
    idx = torch.randperm(batch_size, device=x_text.device)

    x_text  = lam * x_text  + (1 - lam) * x_text[idx]
    x_audio = lam * x_audio + (1 - lam) * x_audio[idx]
    x_vs    = lam * x_vs    + (1 - lam) * x_vs[idx]
    x_vp    = lam * x_vp    + (1 - lam) * x_vp[idx]

    y_a    = torch.zeros(batch_size, num_classes, device=y.device).scatter_(1, y.unsqueeze(1), 1.0)
    y_b    = torch.zeros(batch_size, num_classes, device=y.device).scatter_(1, y[idx].unsqueeze(1), 1.0)
    y_soft = lam * y_a + (1 - lam) * y_b
    return x_text, x_audio, x_vs, x_vp, y_soft


## Model: CalmToConflict_DyadicMulT

4-stream cross-modal Transformer. Within-speaker attention: T ↔ A, T ↔ VS, A ↔ VS. Cross-speaker attention: VS ↔ VP.

In [ ]:
# MAIN MODEL (Dyadic MulT)
class CalmToConflict_DyadicMulT(nn.Module):
    """
    4-stream Multimodal Transformer for dyadic conflict detection.

    Streams
    -------
    T  : Text (EmoRoBERTa), shape [B, T_text,  text_dim]
    A  : Audio (HuBERT-ER), shape [B, T_audio, audio_dim]
    VS : Self  video (FAU+head+gaze+body of the speaker), shape [B, T_vs, video_dim]
    VP : Partner video (partner's 2-second post-utterance reaction), same dim

    Cross-attention graph
    ---------------------
    Within speaker: T ↔ A,  T ↔ VS,  A ↔ VS   
    Cross-speaker:  VS ↔ VP                      (dyadic signal)
    Partner context only:                        VP → T, VP → A kept OFF by default
                                                
    The classifier receives [g_T·T̂, g_A·Â, g_VS·V̂S, g_VP·V̂P] (4×shared_dim).
    """

    def __init__(
        self,
        text_dim:    int   = 768,
        audio_dim:   int   = 768,
        video_dim:   int   = 92,
        shared_dim:  int   = 128,
        num_heads:   int   = 4,
        num_classes: int   = 2,
        dropout:     float = 0.2,
    ):
        super().__init__()
        self.shared_dim = shared_dim

        # --- Input projections (Conv1d = channel-wise linear, no temporal mixing) ---
        self.proj_text  = nn.Conv1d(text_dim,  shared_dim, kernel_size=1)
        self.proj_audio = nn.Conv1d(audio_dim, shared_dim, kernel_size=1)
        self.proj_vs    = nn.Conv1d(video_dim, shared_dim, kernel_size=1)
        self.proj_vp    = nn.Conv1d(video_dim, shared_dim, kernel_size=1)

        # --- Positional encoding (shared across all streams) ---
        self.pos_encoder = PositionalEncoding(d_model=shared_dim, dropout=dropout)

        # ---- Cross-modal attention: within-speaker (6 blocks, same as v1) ----
        self.trans_T_with_A  = CrossModalAttentionBlock(shared_dim, num_heads, dropout)
        self.trans_T_with_VS = CrossModalAttentionBlock(shared_dim, num_heads, dropout)
        self.trans_A_with_T  = CrossModalAttentionBlock(shared_dim, num_heads, dropout)
        self.trans_A_with_VS = CrossModalAttentionBlock(shared_dim, num_heads, dropout)
        self.trans_VS_with_T = CrossModalAttentionBlock(shared_dim, num_heads, dropout)
        self.trans_VS_with_A = CrossModalAttentionBlock(shared_dim, num_heads, dropout)

        # ---- Cross-speaker attention: VS ↔ VP (2 NEW blocks) ----
        # VS queries VP: "given what I said, how did my partner react?"
        self.trans_VS_with_VP = CrossModalAttentionBlock(shared_dim, num_heads, dropout)
        # VP queries VS: "what did the speaker say that triggered this reaction?"
        self.trans_VP_with_VS = CrossModalAttentionBlock(shared_dim, num_heads, dropout)

        # --- Attention pooling (one per stream) ---
        self.pool_T  = AttentionPool(shared_dim)
        self.pool_A  = AttentionPool(shared_dim)
        self.pool_VS = AttentionPool(shared_dim)
        self.pool_VP = AttentionPool(shared_dim)

        # --- Learnable modality gates (sigmoid scalar, one per stream) ---
        self.gate_T  = nn.Parameter(torch.ones(1))
        self.gate_A  = nn.Parameter(torch.ones(1))
        self.gate_VS = nn.Parameter(torch.ones(1))
        self.gate_VP = nn.Parameter(torch.ones(1))

        # --- Classifier head ---
        # Input is 4 × shared_dim after gated concatenation
        self.classifier = nn.Sequential(
            nn.Linear(shared_dim * 4, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(
        self,
        x_text:         torch.Tensor,
        x_audio:        torch.Tensor,
        x_video_self:   torch.Tensor,
        x_video_partner: torch.Tensor,
        use_text:    bool = True,
        use_audio:   bool = True,
        use_vs:      bool = True,
        use_vp:      bool = True,
    ) -> torch.Tensor:

        B   = x_text.size(0)
        dev = x_text.device
        D   = self.shared_dim

        # ---- Helper: project + positional encode (or return zeros) ----
        def _encode(x, proj, use_flag):
            if use_flag:
                out = proj(x.transpose(1, 2)).transpose(1, 2)   # [B, T, D]
                return self.pos_encoder(out)
            return torch.zeros(B, x.size(1), D, device=dev)

        proj_T  = _encode(x_text,          self.proj_text,  use_text)
        proj_A  = _encode(x_audio,         self.proj_audio, use_audio)
        proj_VS = _encode(x_video_self,    self.proj_vs,    use_vs)
        proj_VP = _encode(x_video_partner, self.proj_vp,    use_vp)

        # ---- Within-speaker cross-modal attention ----
        # Text enriched by audio and self-video
        out_T = proj_T
        if use_text and use_audio: out_T = self.trans_T_with_A(out_T, proj_A)
        if use_text and use_vs:    out_T = self.trans_T_with_VS(out_T, proj_VS)

        # Audio enriched by text and self-video
        out_A = proj_A
        if use_audio and use_text: out_A = self.trans_A_with_T(out_A, proj_T)
        if use_audio and use_vs:   out_A = self.trans_A_with_VS(out_A, proj_VS)

        # Self-video enriched by text, audio, AND partner reaction (cross-speaker)
        out_VS = proj_VS
        if use_vs and use_text:  out_VS = self.trans_VS_with_T(out_VS, proj_T)
        if use_vs and use_audio: out_VS = self.trans_VS_with_A(out_VS, proj_A)
        if use_vs and use_vp:    out_VS = self.trans_VS_with_VP(out_VS, proj_VP) 

        # Partner-video enriched by self-video (cross-speaker)
        out_VP = proj_VP
        if use_vp and use_vs: out_VP = self.trans_VP_with_VS(out_VP, proj_VS)    

        # ---- Attention pooling → [B, D] ----
        def _pool(pool_fn, out, use_flag):
            if use_flag:
                return pool_fn(out)
            return torch.zeros(B, D, device=dev)

        pooled_T  = _pool(self.pool_T,  out_T,  use_text)
        pooled_A  = _pool(self.pool_A,  out_A,  use_audio)
        pooled_VS = _pool(self.pool_VS, out_VS, use_vs)
        pooled_VP = _pool(self.pool_VP, out_VP, use_vp)

        # ---- Modality gating ----
        g_T  = torch.sigmoid(self.gate_T)
        g_A  = torch.sigmoid(self.gate_A)
        g_VS = torch.sigmoid(self.gate_VS)
        g_VP = torch.sigmoid(self.gate_VP)

        # ---- Concatenate and classify ----
        fused = torch.cat(
            [g_T * pooled_T, g_A * pooled_A, g_VS * pooled_VS, g_VP * pooled_VP],
            dim=-1
        )  # [B, 4*D]
        return self.classifier(fused)

# LR SCHEDULE (cosine + warm-up)

def get_cosine_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps):
    def lr_lambda(step):
        if step < num_warmup_steps:
            return float(step) / float(max(1, num_warmup_steps))
        progress = float(step - num_warmup_steps) / float(
            max(1, num_training_steps - num_warmup_steps)
        )
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)



## Training Loop & Ablation Runner

Trains across all seeds and experiments, tracks the globally best weights per experiment (by macro F1), and saves one `.pt` file each.

In [ ]:
# TRAINING LOOP
def train_and_evaluate(
    model,
    train_loader:    DataLoader,
    test_loader:     DataLoader,
    device,
    train_df:        pd.DataFrame,
    epochs:          int   = 25,
    lr:              float = 1e-4,
    use_text:        bool  = True,
    use_audio:       bool  = True,
    use_vs:          bool  = True,
    use_vp:          bool  = True,
    mixup_alpha:     float = 0.2,
    label_smooth:    float = 0.05,
    experiment_name: str   = "Dyadic_MulT",
    plot_dir:        str   = None,
):
    print(f"\n{'='*55}")
    print(f"STARTING EXPERIMENT: {experiment_name}")
    print(f"Modalities -> Text: {use_text} | Audio: {use_audio} | "
          f"SelfVideo: {use_vs} | PartnerVideo: {use_vp}")
    print(f"{'='*55}\n")

    model = model.to(device)

    # Class-weighted loss (handles ~1:1 balance, but keeps it robust)
    class_counts = train_df["label"].value_counts().sort_index().values
    weights = torch.tensor(1.0 / class_counts.astype(float), dtype=torch.float32)
    weights = (weights / weights.sum()).to(device)

    total_steps  = epochs * len(train_loader)
    warmup_steps = len(train_loader)
    optimizer    = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler    = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    train_losses, val_losses, val_accuracies = [], [], []
    best_val_loss  = float("inf")
    best_model_wts = copy.deepcopy(model.state_dict())
    best_epoch     = 1
    patience       = 10
    patience_ctr   = 0

    for epoch in range(epochs):
        # ---- Training ----
        model.train()
        running_loss = 0.0

        for batch_T, batch_A, batch_VS, batch_VP, batch_y in train_loader:
            batch_T  = batch_T.to(device)
            batch_A  = batch_A.to(device)
            batch_VS = batch_VS.to(device)
            batch_VP = batch_VP.to(device)
            batch_y  = batch_y.to(device)

            # MixUp across all 4 streams
            batch_T, batch_A, batch_VS, batch_VP, soft_y = mixup_batch(
                batch_T, batch_A, batch_VS, batch_VP, batch_y, alpha=mixup_alpha
            )

            # Label smoothing
            if label_smooth > 0:
                soft_y = (1 - label_smooth) * soft_y + label_smooth / soft_y.size(1)

            logits = model(batch_T, batch_A, batch_VS, batch_VP,
                           use_text, use_audio, use_vs, use_vp)

            log_probs = F.log_softmax(logits, dim=-1)
            weighted_log = log_probs * weights.unsqueeze(0)
            loss = -(soft_y * weighted_log).sum(dim=-1).mean()

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            running_loss += loss.item()

        epoch_train_loss = running_loss / len(train_loader)
        train_losses.append(epoch_train_loss)

        # ---- Validation ----
        model.eval()
        running_val_loss = 0.0
        val_preds, val_targets = [], []

        with torch.no_grad():
            for batch_T, batch_A, batch_VS, batch_VP, batch_y in test_loader:
                batch_T  = batch_T.to(device)
                batch_A  = batch_A.to(device)
                batch_VS = batch_VS.to(device)
                batch_VP = batch_VP.to(device)
                batch_y  = batch_y.to(device)

                logits = model(batch_T, batch_A, batch_VS, batch_VP,
                               use_text, use_audio, use_vs, use_vp)

                y_onehot = torch.zeros_like(logits).scatter_(1, batch_y.unsqueeze(1), 1.0)
                log_probs = F.log_softmax(logits, dim=-1)
                val_loss  = -(y_onehot * (log_probs * weights.unsqueeze(0))).sum(dim=-1).mean()
                running_val_loss += val_loss.item()

                _, predicted = torch.max(logits, 1)
                val_preds.extend(predicted.cpu().numpy())
                val_targets.extend(batch_y.cpu().numpy())

        epoch_val_loss = running_val_loss / len(test_loader)
        epoch_val_acc  = accuracy_score(val_targets, val_preds)
        val_losses.append(epoch_val_loss)
        val_accuracies.append(epoch_val_acc)

        if (epoch + 1) % 5 == 0 or epoch == 0:
            current_lr = optimizer.param_groups[0]["lr"]
            print(
                f"Epoch [{epoch+1:02d}/{epochs}] | "
                f"Train Loss: {epoch_train_loss:.4f} | "
                f"Val Loss: {epoch_val_loss:.4f} | "
                f"Val Acc: {epoch_val_acc*100:.2f}% | "
                f"LR: {current_lr:.2e}"
            )

        if epoch_val_loss < best_val_loss:
            best_val_loss  = epoch_val_loss
            best_model_wts = copy.deepcopy(model.state_dict())
            best_epoch     = epoch + 1
            patience_ctr   = 0
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print(f"\n  Early stopping at epoch {epoch+1} (no improvement for {patience} epochs).")
                break

    print(f"\nRestoring best weights from Epoch {best_epoch} (Val Loss: {best_val_loss:.4f})")
    model.load_state_dict(best_model_wts)

    # ---- Final evaluation ----
    model.eval()
    final_preds, final_targets = [], []
    with torch.no_grad():
        for batch_T, batch_A, batch_VS, batch_VP, batch_y in test_loader:
            logits = model(
                batch_T.to(device), batch_A.to(device),
                batch_VS.to(device), batch_VP.to(device),
                use_text, use_audio, use_vs, use_vp
            )
            _, predicted = torch.max(logits, 1)
            final_preds.extend(predicted.cpu().numpy())
            final_targets.extend(batch_y.numpy())

    best_acc = accuracy_score(final_targets, final_preds)
    f1       = f1_score(final_targets, final_preds, average="macro")

    print("\n" + "="*50)
    print(f"FINAL RESULTS: {experiment_name}")
    print("="*50)
    print(f"Accuracy : {best_acc*100:.2f}%")
    print(f"F1 (macro): {f1:.4f}")
    print("-"*50)
    print(classification_report(
        final_targets, final_preds,
        target_names=["Calm (0)", "Conflict (1)"]
    ))

    # Print learned gates when all 4 streams active
    if use_text and use_audio and use_vs and use_vp:
        gT  = torch.sigmoid(model.gate_T).item()
        gA  = torch.sigmoid(model.gate_A).item()
        gVS = torch.sigmoid(model.gate_VS).item()
        gVP = torch.sigmoid(model.gate_VP).item()
        print(f"Learned modality gates — Text: {gT:.3f} | Audio: {gA:.3f} | "
              f"SelfVideo: {gVS:.3f} | PartnerVideo: {gVP:.3f}")

    # ── Plots (shared helper) ──────────────────────────────────────────────
    save_training_plots(
        experiment_name, train_losses, val_losses, val_accuracies,
        best_epoch, final_preds, final_targets
    )
    return best_acc, f1

if __name__ == "__main__":

    # ── Load data once (seeds only affect weight init / shuffle) ─────────────
    df = pd.read_csv(PATHS["csv"])

    print("Loading embeddings...")
    text_dict          = load_embedding("text")
    audio_dict         = load_embedding("audio")
    self_video_dict    = load_embedding("video_self")
    partner_video_dict = load_embedding("video_partner")

    dyadic_ids = set(df["sample_id"])
    text_ids   = set(text_dict.keys())
    missing_text = dyadic_ids - text_ids
    if missing_text:
        print(f"WARNING: {len(missing_text)} dyadic sample_ids missing from text embeddings.")

    # Fixed group-based split (seed=42 is for the split itself, not the model)
    train_df, test_df = group_split(df, test_size=0.20, seed=42)
    print(f"\nTrain: {len(train_df)} utterances from {train_df['file_id'].nunique()} unique videos")
    print(f"Test : {len(test_df)} utterances from {test_df['file_id'].nunique()} unique videos")
    print(f"Train label dist: {train_df['label'].value_counts().to_dict()}")
    print(f"Test  label dist: {test_df['label'].value_counts().to_dict()}")

    train_dataset = DyadicSequenceDataset(train_df, text_dict, audio_dict,
                                          self_video_dict, partner_video_dict)
    test_dataset  = DyadicSequenceDataset(test_df,  text_dict, audio_dict,
                                          self_video_dict, partner_video_dict)

    VIDEO_DIM = next(iter(self_video_dict.values())).shape[-1]
    AUDIO_DIM = next(iter(audio_dict.values())).shape[-1]
    TEXT_DIM  = next(iter(text_dict.values())).shape[-1]
    print(f"\nAuto-detected dims -> Text: {TEXT_DIM} | Audio: {AUDIO_DIM} | Video: {VIDEO_DIM}")

    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,
                              num_workers=8, pin_memory=True)
    test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False,
                              num_workers=8, pin_memory=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}\n")

    # =================================================================
    # ABLATION STUDY — 9 experiments × 5 seeds
    # =================================================================
    experiments = [
        # (name,                 use_T,  use_A,  use_VS, use_VP)
        ("Full_Dyadic",          True,  True,   True,   True),
        ("No_Partner",           True,  True,   True,   False),
        ("SelfVideo_PartnerVideo", False, False, True,   True),
        ("No_SelfVideo",         True,  True,   False,  True),
        ("No_Audio",             True,  False,  True,   True),
        ("No_Text",              False, True,   True,   True),
        ("Only_PartnerVideo",    False, False,  False,  True),
        ("Only_SelfVideo",       False, False,  True,   False),
        ("Audio_SelfVideo",      False, True,   True,   False),
    ]

    save_dir    = PATHS["models"]
    results_dir = PATHS["results"]

    # Accumulate per-seed results: {exp_name: {"acc": [...], "f1": [...]}}
    multi_seed_results = {exp_name: {"acc": [], "f1": []} for exp_name, *_ in experiments}
    # Track the globally best model weights per experiment (across seeds)
    best_weights_per_exp = {exp_name: {"val_loss": float("inf"), "state_dict": None}
                            for exp_name, *_ in experiments}

    for seed in RANDOM_SEEDS:
        print(f"\n{'#'*65}")
        print(f"  SEED {seed}")
        print(f"{'#'*65}")
        set_seed(seed)

        for exp_name, use_t, use_a, use_vs, use_vp in experiments:
            model = CalmToConflict_DyadicMulT(
                text_dim=TEXT_DIM, audio_dim=AUDIO_DIM, video_dim=VIDEO_DIM,
                shared_dim=128, num_heads=4
            ).to(device)

            acc, f1 = train_and_evaluate(
                model, train_loader, test_loader, device, train_df,
                epochs=25, lr=1e-4,
                use_text=use_t, use_audio=use_a, use_vs=use_vs, use_vp=use_vp,
                mixup_alpha=0.2, label_smooth=0.05,
                experiment_name=f"{exp_name}_seed{seed}",
            )
            multi_seed_results[exp_name]["acc"].append(acc)
            multi_seed_results[exp_name]["f1"].append(f1)

            # ── Keep absolute best model weights across all seeds ────────────
            # train_and_evaluate already restores the best epoch weights into model
            # We compare by val F1 as the primary metric for "best model"
            current_best_f1 = max(multi_seed_results[exp_name]["f1"])
            if f1 >= current_best_f1:
                best_weights_per_exp[exp_name]["state_dict"] = {
                    k: v.cpu().clone() for k, v in model.state_dict().items()
                }
                best_weights_per_exp[exp_name]["seed"] = seed
                best_weights_per_exp[exp_name]["acc"] = acc
                best_weights_per_exp[exp_name]["f1"] = f1

    # ── Save best weights (one .pt per experiment) ───────────────────────────
    for exp_name, info in best_weights_per_exp.items():
        if info["state_dict"] is not None:
            save_path = os.path.join(save_dir, f"{exp_name}_best_weights.pt")
            torch.save(info["state_dict"], save_path)
            print(f"Saved best weights for {exp_name} "
                  f"(seed={info['seed']}, F1={info['f1']:.4f}) -> {save_path}")

    # ── Aggregate mean ± std ─────────────────────────────────────────────────
    summary = {}
    for exp_name, vals in multi_seed_results.items():
        acc_arr = np.array(vals["acc"])
        f1_arr  = np.array(vals["f1"])
        summary[exp_name] = {
            "acc_mean":  float(acc_arr.mean()),
            "acc_std":   float(acc_arr.std()),
            "f1_mean":   float(f1_arr.mean()),
            "f1_std":    float(f1_arr.std()),
            "acc_per_seed": [float(x) for x in acc_arr],
            "f1_per_seed":  [float(x) for x in f1_arr],
            "seeds":     RANDOM_SEEDS,
        }

    # ── Save results JSON ────────────────────────────────────────────────────
    json_path = os.path.join(results_dir, "dyadic_mult_results.json")
    with open(json_path, "w") as f:
        json.dump(summary, f, indent=2)
    print(f"\nResults saved to {json_path}")

    # ── Print summary table ───────────────────────────────────────────────────
    print("\n" + "="*75)
    print("ABLATION SUMMARY — Dyadic MulT  (mean ± std over seeds: "
          + str(RANDOM_SEEDS) + ")")
    print("="*75)
    print(f"{'Experiment':<30} {'Acc (%)':>14} {'F1 (macro)':>16}")
    print("-"*75)
    for name, r in summary.items():
        print(f"{name:<30} {r['acc_mean']*100:>7.2f} ± {r['acc_std']*100:<5.2f} "
              f"{r['f1_mean']:>8.4f} ± {r['f1_std']:.4f}")
    print("="*75)


---
# Section 3 — Visual Feature Ablation

Fine-grained ablation within the 92-dim visual stream, decomposed into four sub-features: FAU (24d), head orientation (3d), gaze (2d), and body pose (63d). Runs 10 experiments × `RANDOM_SEEDS` to identify which physical behaviors drive predictions.

## Feature Boundary Loader

Reads the slice indices produced during video extraction and returns them as a `dict` of `(start, end)` tuples for direct tensor indexing.

In [ ]:
def load_feature_dims(json_path: str = "embeddings/video_feature_dims.json") -> dict:
    """
    Loads the feature boundary dict produced by video_extraction_v2.py.
    Returns slices as Python slices for direct tensor indexing.

    Example output:
        {
          'fau':  (0, 24),
          'head': (24, 30),
          'gaze': (30, 36),
          'body': (36, 99),
        }
    """
    if not os.path.exists(json_path):
        raise FileNotFoundError(
            f"{json_path} not found.\n"
            "Run video_extraction_v2.py first to generate feature boundary metadata."
        )
    with open(json_path) as f:
        meta = json.load(f)

    return {
        "fau":  tuple(meta["fau_slice"]),
        "head": tuple(meta["head_slice"]),
        "gaze": tuple(meta["gaze_slice"]),
        "body": tuple(meta["body_slice"]),
    }

## Dataset

Pre-slices the 92-dim video tensors into the 4 sub-streams at dataset construction time, avoiding repeated slicing overhead in the training loop.

In [ ]:
class VisualAblationDataset(Dataset):
    """
    Returns pre-split video sub-streams so the model can ablate them
    without any per-epoch slicing overhead.

    Items returned: (vs_fau, vs_head, vs_gaze, vs_body,
                     vp_fau, vp_head, vp_gaze, vp_body, label)
    """

    def __init__(self, df,
                 self_video_dict, partner_video_dict,
                 feature_slices: dict):
        self.samples = []
        missing = 0
        fs = feature_slices   # shorthand

        for _, row in df.iterrows():
            sid = row["sample_id"]
            if sid not in self_video_dict:
                missing += 1
                continue

            vs = self_video_dict[sid]                                    # [T, D]
            vp = partner_video_dict.get(sid, torch.zeros_like(vs))

            self.samples.append({
                "label":   int(row["label"]),
                # --- speaker sub-streams ---
                "vs_fau":  vs[:, fs["fau"][0]:fs["fau"][1]],
                "vs_head": vs[:, fs["head"][0]:fs["head"][1]],
                "vs_gaze": vs[:, fs["gaze"][0]:fs["gaze"][1]],
                "vs_body": vs[:, fs["body"][0]:fs["body"][1]],
                # --- partner sub-streams ---
                "vp_fau":  vp[:, fs["fau"][0]:fs["fau"][1]],
                "vp_head": vp[:, fs["head"][0]:fs["head"][1]],
                "vp_gaze": vp[:, fs["gaze"][0]:fs["gaze"][1]],
                "vp_body": vp[:, fs["body"][0]:fs["body"][1]],
            })

        if missing:
            print(f"  [Dataset] Skipped {missing} samples missing from video dicts.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        return (
            item["vs_fau"].clone().float(),
            item["vs_head"].clone().float(),
            item["vs_gaze"].clone().float(),
            item["vs_body"].clone().float(),
            item["vp_fau"].clone().float(),
            item["vp_head"].clone().float(),
            item["vp_gaze"].clone().float(),
            item["vp_body"].clone().float(),
            torch.tensor(item["label"], dtype=torch.long),
        )


## MixUp (8 sub-streams)

In [ ]:
def mixup_batch(streams: list, y: torch.Tensor, alpha: float = 0.2):
    """
    streams : list of 8 tensors [B, T, D_sub]
    Returns mixed streams (list) and soft labels.
    """
    B = y.size(0)
    NC = 2

    if alpha <= 0:
        y_soft = torch.zeros(B, NC, device=y.device).scatter_(1, y.unsqueeze(1), 1.0)
        return streams, y_soft

    lam = np.random.beta(alpha, alpha)
    lam = max(lam, 1 - lam)
    idx = torch.randperm(B, device=streams[0].device)

    mixed = [lam * s + (1 - lam) * s[idx] for s in streams]

    y_a    = torch.zeros(B, NC, device=y.device).scatter_(1, y.unsqueeze(1), 1.0)
    y_b    = torch.zeros(B, NC, device=y.device).scatter_(1, y[idx].unsqueeze(1), 1.0)
    y_soft = lam * y_a + (1 - lam) * y_b
    return mixed, y_soft

## Model: VisualDyadicMulT

Accepts up to 8 visual sub-streams (4 speaker + 4 partner). Intra-speaker attention: VS_i ← VS_j for all active i ≠ j. Cross-speaker attention: VS_i ↔ VP_i.

In [ ]:
"""
CalmToConflict MulT — Visual Feature Ablation
=================================================

Starting point
--------------
v1's ablation showed that SelfVideo + PartnerVideo is a very 
strong configuration.

Research question answered here
--------------------------------
Within the visual modality (FAU=24, head=3, gaze=2, body=63 dims),
WHICH feature type(s) drive the cross-speaker gain?

Approach
--------
No re-extraction is needed. The combined video tensors [T, D_total] have
features concatenated in a fixed order: [FAU | head | gaze | body].
The boundary indices are read from embeddings/video_feature_dims.json
(generated in Section 1).

At dataset time, each tensor is sliced into 4 sub-tensors. The model
receives separate projection heads for each, processes them in parallel,
and the ablation flag masks entire sub-streams to zero before projection.

Model architecture
------------------
Same cross-speaker MulT as v1, but the "video_self" and "video_partner"
streams are each split into 4 sub-streams before projection:
  VS_FAU, VS_head, VS_gaze, VS_body
  VP_FAU, VP_head, VP_gaze, VP_body

All 8 sub-streams share the same CrossModalAttentionBlock instances
(weight sharing via stream type), with the same gating mechanism.

Ablation experiments (10 total)
--------------------------------
 1. Full_Visual_Dyadic       — all 8 sub-streams 
 2. FAU_Only                 — VS_FAU + VP_FAU
 3. Head_Only                — VS_head + VP_head
 4. Gaze_Only                — VS_gaze + VP_gaze
 5. Body_Only                — VS_body + VP_body
 6. FAU_Head                 — FAU + head (face+orientation)
 7. FAU_Gaze                 — FAU + gaze (face+eye contact)
 8. FAU_Body                 — FAU + body (face+posture)
 9. NoBody                   — FAU + head + gaze (all face streams)
10. Body_Partner_Only        — VS_body + VP_FAU  (posture self, face partner)

The last experiment is theoretically motivated: body posture of the speaker
(VS_body) and partner facial expression (VP_FAU) may together form the
strongest dyadic conflict signal.
"""

class VisualDyadicMulT(nn.Module):
    """
    Processes up to 8 visual sub-streams (4 speaker + 4 partner).
    Each sub-stream has its own projection layer.
    Cross-speaker attention: VS_i attends to VP_i for each active feature i.
    Intra-speaker attention: Each VS_i attends to all other active VS_j.

    Classifier input: concat of all active pooled sub-stream vectors.
    Classifier size adapts dynamically to the number of active streams.
    """

    STREAM_KEYS = ["fau", "head", "gaze", "body"]

    def __init__(
        self,
        feature_dims: dict,   # {"fau": D_fau, "head": D_head, ...}
        shared_dim:   int   = 64,
        num_heads:    int   = 4,
        num_classes:  int   = 2,
        dropout:      float = 0.2,
    ):
        super().__init__()
        self.shared_dim   = shared_dim
        self.feature_dims = feature_dims
        self.keys         = self.STREAM_KEYS

        # --- Per-sub-stream projection (speaker and partner share weights) ---
        self.proj = nn.ModuleDict({
            k: nn.Conv1d(feature_dims[k], shared_dim, kernel_size=1)
            for k in self.keys
        })

        # --- Positional encoding (shared) ---
        self.pos_encoder = PositionalEncoding(d_model=shared_dim, dropout=dropout)

        # --- Intra-speaker cross-feature attention (VS_i ← VS_j) ---
        # One block per (query_key, kv_key) ordered pair, shared across speakers
        self.intra_attn = nn.ModuleDict({
            f"{q}_{kv}": CrossModalAttentionBlock(shared_dim, num_heads, dropout)
            for q  in self.keys
            for kv in self.keys
            if q != kv
        })

        # --- Cross-speaker attention: VS_i ← VP_i (same feature type) ---
        self.cross_self_to_partner = nn.ModuleDict({
            k: CrossModalAttentionBlock(shared_dim, num_heads, dropout)
            for k in self.keys
        })
        self.cross_partner_to_self = nn.ModuleDict({
            k: CrossModalAttentionBlock(shared_dim, num_heads, dropout)
            for k in self.keys
        })

        # --- Attention pooling and gating (speaker + partner, per feature) ---
        self.pool_vs = nn.ModuleDict({k: AttentionPool(shared_dim) for k in self.keys})
        self.pool_vp = nn.ModuleDict({k: AttentionPool(shared_dim) for k in self.keys})
        self.gate_vs = nn.ParameterDict({k: nn.Parameter(torch.ones(1)) for k in self.keys})
        self.gate_vp = nn.ParameterDict({k: nn.Parameter(torch.ones(1)) for k in self.keys})

        # --- Classifier (size depends on active streams, built dynamically) ---
        # Max possible: 8 streams × shared_dim
        self._max_fused_dim = shared_dim * len(self.keys) * 2
        self.classifier_hidden = nn.Sequential(
            nn.Linear(self._max_fused_dim, 256), nn.BatchNorm1d(256), nn.GELU(), nn.Dropout(0.4),
            nn.Linear(256, 128),                 nn.GELU(),            nn.Dropout(0.3),
        )
        self.classifier_head = nn.Linear(128, num_classes)

        # For dynamic fused dim, we project any active combination to the fixed max_fused_dim
        self.input_proj = nn.Linear(self._max_fused_dim, self._max_fused_dim)

    def forward(
        self,
        sub_streams: dict,  # {"vs_fau": tensor, "vs_head": ..., ..., "vp_fau": ..., ...}
        active_vs:   set,   # e.g. {"fau", "body"}
        active_vp:   set,   # e.g. {"fau"}
    ) -> torch.Tensor:
        B   = next(iter(sub_streams.values())).size(0)
        dev = next(iter(sub_streams.values())).device
        D   = self.shared_dim

        def _encode(x, key):
            out = self.proj[key](x.transpose(1, 2)).transpose(1, 2)
            return self.pos_encoder(out)

        # --- Project and encode active sub-streams ---
        enc_vs = {
            k: _encode(sub_streams[f"vs_{k}"], k)
            for k in active_vs
        }
        enc_vp = {
            k: _encode(sub_streams[f"vp_{k}"], k)
            for k in active_vp
        }

        # --- Intra-speaker: each VS_i attends to all other active VS_j ---
        out_vs = {k: enc_vs[k].clone() for k in active_vs}
        for q_key in active_vs:
            for kv_key in active_vs:
                if q_key != kv_key:
                    block_key = f"{q_key}_{kv_key}"
                    out_vs[q_key] = self.intra_attn[block_key](out_vs[q_key], enc_vs[kv_key])

        # --- Cross-speaker: VS_i ← VP_i and VP_i ← VS_i ---
        out_vp = {k: enc_vp[k].clone() for k in active_vp}
        for k in active_vs:
            if k in active_vp:
                out_vs[k] = self.cross_self_to_partner[k](out_vs[k], enc_vp[k])
        for k in active_vp:
            if k in active_vs:
                out_vp[k] = self.cross_partner_to_self[k](out_vp[k], enc_vs[k])

        # --- Pool and gate ---
        pooled_parts = []
        for k in self.keys:
            if k in active_vs:
                g  = torch.sigmoid(self.gate_vs[k])
                pv = self.pool_vs[k](out_vs[k])
                pooled_parts.append(g * pv)
            else:
                pooled_parts.append(torch.zeros(B, D, device=dev))

            if k in active_vp:
                g  = torch.sigmoid(self.gate_vp[k])
                pp = self.pool_vp[k](out_vp[k])
                pooled_parts.append(g * pp)
            else:
                pooled_parts.append(torch.zeros(B, D, device=dev))

        # pooled_parts: 8 vectors of shape [B, D] (some may be zero)
        fused = torch.cat(pooled_parts, dim=-1)   # [B, 8*D]
        fused = self.input_proj(fused)
        return self.classifier_head(self.classifier_hidden(fused))



def get_cosine_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps):
    def lr_lambda(step):
        if step < num_warmup_steps:
            return float(step) / float(max(1, num_warmup_steps))
        progress = float(step - num_warmup_steps) / float(
            max(1, num_training_steps - num_warmup_steps)
        )
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


## Training Loop & Ablation Runner

In [ ]:
# TRAINING LOOP

def train_and_evaluate(
    model,
    train_loader,
    test_loader,
    device,
    train_df,
    active_vs: set,
    active_vp: set,
    epochs:          int   = 25,
    lr:              float = 1e-4,
    mixup_alpha:     float = 0.2,
    label_smooth:    float = 0.05,
    experiment_name: str   = "VisualDyadic",
    plot_dir:        str   = None,
):
    print(f"\n{'='*60}")
    print(f"EXPERIMENT: {experiment_name}")
    print(f"Active self   streams: {sorted(active_vs) if active_vs else '(none)'}")
    print(f"Active partner streams: {sorted(active_vp) if active_vp else '(none)'}")
    print(f"{'='*60}\n")

    model = model.to(device)

    class_counts = train_df["label"].value_counts().sort_index().values
    weights      = torch.tensor(1.0 / class_counts.astype(float), dtype=torch.float32)
    weights      = (weights / weights.sum()).to(device)

    total_steps  = epochs * len(train_loader)
    warmup_steps = len(train_loader)
    optimizer    = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler    = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    stream_keys   = ["vs_fau", "vs_head", "vs_gaze", "vs_body",
                     "vp_fau", "vp_head", "vp_gaze", "vp_body"]

    train_losses, val_losses, val_accuracies = [], [], []
    best_val_loss  = float("inf")
    best_model_wts = copy.deepcopy(model.state_dict())
    best_epoch     = 1
    patience       = 6
    patience_ctr   = 0

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for batch in train_loader:
            *stream_tensors, batch_y = batch
            stream_tensors = [t.to(device) for t in stream_tensors]
            batch_y        = batch_y.to(device)

            mixed, soft_y = mixup_batch(stream_tensors, batch_y, alpha=mixup_alpha)
            if label_smooth > 0:
                soft_y = (1 - label_smooth) * soft_y + label_smooth / soft_y.size(1)

            sub_streams = dict(zip(stream_keys, mixed))
            logits       = model(sub_streams, active_vs, active_vp)

            log_probs    = F.log_softmax(logits, dim=-1)
            loss         = -(soft_y * (log_probs * weights.unsqueeze(0))).sum(dim=-1).mean()

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            running_loss += loss.item()

        epoch_train_loss = running_loss / len(train_loader)
        train_losses.append(epoch_train_loss)

        model.eval()
        running_val_loss = 0.0
        val_preds, val_targets = [], []

        with torch.no_grad():
            for batch in test_loader:
                *stream_tensors, batch_y = batch
                stream_tensors = [t.to(device) for t in stream_tensors]
                batch_y        = batch_y.to(device)

                sub_streams = dict(zip(stream_keys, stream_tensors))
                logits       = model(sub_streams, active_vs, active_vp)

                y_onehot  = torch.zeros_like(logits).scatter_(1, batch_y.unsqueeze(1), 1.0)
                log_probs = F.log_softmax(logits, dim=-1)
                val_loss  = -(y_onehot * (log_probs * weights.unsqueeze(0))).sum(dim=-1).mean()
                running_val_loss += val_loss.item()

                _, predicted = torch.max(logits, 1)
                val_preds.extend(predicted.cpu().numpy())
                val_targets.extend(batch_y.cpu().numpy())

        epoch_val_loss = running_val_loss / len(test_loader)
        epoch_val_acc  = accuracy_score(val_targets, val_preds)
        val_losses.append(epoch_val_loss)
        val_accuracies.append(epoch_val_acc)

        if (epoch + 1) % 5 == 0 or epoch == 0:
            current_lr = optimizer.param_groups[0]["lr"]
            print(f"Epoch [{epoch+1:02d}/{epochs}] | "
                  f"Train Loss: {epoch_train_loss:.4f} | "
                  f"Val Loss: {epoch_val_loss:.4f} | "
                  f"Val Acc: {epoch_val_acc*100:.2f}% | "
                  f"LR: {current_lr:.2e}")

        if epoch_val_loss < best_val_loss:
            best_val_loss  = epoch_val_loss
            best_model_wts = copy.deepcopy(model.state_dict())
            best_epoch     = epoch + 1
            patience_ctr   = 0
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print(f"\n  Early stopping at epoch {epoch+1}.")
                break

    print(f"\nRestoring best weights from Epoch {best_epoch} (Val Loss: {best_val_loss:.4f})")
    model.load_state_dict(best_model_wts)

    model.eval()
    final_preds, final_targets = [], []
    with torch.no_grad():
        for batch in test_loader:
            *stream_tensors, batch_y = batch
            sub_streams = dict(zip(stream_keys, [t.to(device) for t in stream_tensors]))
            logits       = model(sub_streams, active_vs, active_vp)
            _, predicted = torch.max(logits, 1)
            final_preds.extend(predicted.cpu().numpy())
            final_targets.extend(batch_y.numpy())

    best_acc = accuracy_score(final_targets, final_preds)
    f1       = f1_score(final_targets, final_preds, average="macro")

    print("\n" + "="*50)
    print(f"FINAL RESULTS: {experiment_name}")
    print("="*50)
    print(f"Accuracy : {best_acc*100:.2f}%")
    print(f"F1 (macro): {f1:.4f}")
    print("-"*50)
    print(classification_report(final_targets, final_preds,
                                target_names=["Calm (0)", "Conflict (1)"]))

    # Print learned gates for active sub-streams
    gate_str = []
    for k in ["fau", "head", "gaze", "body"]:
        if k in active_vs:
            gate_str.append(f"VS_{k}: {torch.sigmoid(model.gate_vs[k]).item():.3f}")
        if k in active_vp:
            gate_str.append(f"VP_{k}: {torch.sigmoid(model.gate_vp[k]).item():.3f}")
    if gate_str:
        print("Learned gates — " + " | ".join(gate_str))

    # Plots
    plot_dir = plot_dir or PATHS["plots"]
    os.makedirs(plot_dir, exist_ok=True)
    epochs_run = len(train_losses)

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(range(1, epochs_run + 1), train_losses,
             label="Train Loss", color="steelblue", linewidth=2)
    plt.plot(range(1, epochs_run + 1), val_losses,
             label="Val Loss", color="tomato", linewidth=2, linestyle="--")
    plt.axvline(x=best_epoch, color="green", linestyle=":", label=f"Best Epoch {best_epoch}")
    plt.title(f"Loss: {experiment_name}", fontsize=12)
    plt.xlabel("Epochs"); plt.ylabel("Loss")
    plt.legend(); plt.grid(True, alpha=0.3)

    plt.subplot(1, 2, 2)
    plt.plot(range(1, epochs_run + 1), [a * 100 for a in val_accuracies],
             label="Val Accuracy", color="mediumseagreen", linewidth=2)
    plt.axvline(x=best_epoch, color="red", linestyle=":", label=f"Best Epoch {best_epoch}")
    plt.title(f"Accuracy: {experiment_name}", fontsize=12)
    plt.xlabel("Epochs"); plt.ylabel("Accuracy (%)")
    plt.legend(); plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(plot_dir, f"{experiment_name}_curves.png"), dpi=200)
    try:
        plt.show(block=False); plt.pause(2)
    except Exception:
        pass
    plt.close()

    cm = confusion_matrix(final_targets, final_preds)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Predicted Calm", "Predicted Conflict"],
                yticklabels=["Actual Calm",    "Actual Conflict"],
                annot_kws={"size": 14})
    plt.title(f"Confusion Matrix: {experiment_name}", fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(plot_dir, f"{experiment_name}_cm.png"), dpi=200)
    try:
        plt.show(block=False); plt.pause(2)
    except Exception:
        pass
    plt.close()

    print(f"Plots saved to {plot_dir}/")
    return best_acc, f1

if __name__ == "__main__":

    # ── Load data once ───────────────────────────────────────────────────────
    with open("embeddings/video_feature_dims.json") as f:
        meta = json.load(f)
    dims = load_feature_dims()

    df = pd.read_csv(PATHS["csv"])
    print("Loading embeddings for Visual Ablation...")
    audio_dict         = load_embedding("audio")
    self_video_dict    = load_embedding("video_self")
    partner_video_dict = load_embedding("video_partner")

    train_df, test_df = group_split(df, test_size=0.20, seed=42)

    train_dataset = VisualAblationDataset(train_df, audio_dict, self_video_dict,
                                          partner_video_dict, dims)
    test_dataset  = VisualAblationDataset(test_df,  audio_dict, self_video_dict,
                                          partner_video_dict, dims)

    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,
                              num_workers=8, pin_memory=True)
    test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False,
                              num_workers=8, pin_memory=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    AUDIO_DIM = next(iter(audio_dict.values())).shape[-1]
    print(f"Device: {device} | Audio dim: {AUDIO_DIM}")

    # Visual ablation experiment definitions
    # Each entry: (name, vs_features_set, vp_features_set)
    vis_experiments = [
        ("Full_Visual_Dyadic",      {"fau","head","gaze","body"}, {"fau","head","gaze","body"}),
        ("FAU_Only",                {"fau"},                      {"fau"}),
        ("Head_Only",               {"head"},                     {"head"}),
        ("Gaze_Only",               {"gaze"},                     {"gaze"}),
        ("Body_Only",               {"body"},                     {"body"}),
        ("FAU_Head",                {"fau","head"},               {"fau","head"}),
        ("FAU_Gaze",                {"fau","gaze"},               {"fau","gaze"}),
        ("FAU_Body",                {"fau","body"},               {"fau","body"}),
        ("Face_Only",               {"fau","head","gaze"},        {"fau","head","gaze"}),
        ("Body_Speaker_FAU_Partner",{"body"},                     {"fau"}),
    ]

    save_dir    = PATHS["models"]
    results_dir = PATHS["results"]

    multi_seed_results_vis = {exp_name: {"acc": [], "f1": []}
                              for exp_name, *_ in vis_experiments}
    best_weights_vis = {exp_name: {"f1": -1.0, "state_dict": None}
                        for exp_name, *_ in vis_experiments}

    for seed in RANDOM_SEEDS:
        print(f"\n{'#'*65}")
        print(f"  VISUAL ABLATION — SEED {seed}")
        print(f"{'#'*65}")
        set_seed(seed)

        for exp_name, active_vs, active_vp in vis_experiments:
            vs_dim = sum(dims[f"{k}_dim"] for k in active_vs)
            vp_dim = sum(dims[f"{k}_dim"] for k in active_vp)
            model = VisualDyadicMulT(
                audio_dim=AUDIO_DIM,
                vs_feature_keys=active_vs,
                vp_feature_keys=active_vp,
                dims=dims,
                shared_dim=64,
                num_heads=4,
            ).to(device)

            acc, f1 = train_and_evaluate(
                model, train_loader, test_loader, device, train_df,
                active_vs=active_vs, active_vp=active_vp,
                epochs=25, lr=1e-4,
                mixup_alpha=0.2, label_smooth=0.05,
                experiment_name=f"{exp_name}_seed{seed}",
            )
            multi_seed_results_vis[exp_name]["acc"].append(acc)
            multi_seed_results_vis[exp_name]["f1"].append(f1)

            if f1 >= best_weights_vis[exp_name]["f1"]:
                best_weights_vis[exp_name] = {
                    "f1": f1, "acc": acc, "seed": seed,
                    "state_dict": {k: v.cpu().clone() for k, v in model.state_dict().items()}
                }

    # ── Save best weights ────────────────────────────────────────────────────
    for exp_name, info in best_weights_vis.items():
        if info["state_dict"] is not None:
            save_path = os.path.join(save_dir, f"{exp_name}_best_weights.pt")
            torch.save(info["state_dict"], save_path)
            print(f"Saved {exp_name} best weights (seed={info['seed']}, "
                  f"F1={info['f1']:.4f}) -> {save_path}")

    # ── Aggregate & save JSON ────────────────────────────────────────────────
    summary_vis = {}
    for exp_name, vals in multi_seed_results_vis.items():
        acc_arr = np.array(vals["acc"])
        f1_arr  = np.array(vals["f1"])
        summary_vis[exp_name] = {
            "acc_mean": float(acc_arr.mean()), "acc_std": float(acc_arr.std()),
            "f1_mean":  float(f1_arr.mean()),  "f1_std":  float(f1_arr.std()),
            "acc_per_seed": [float(x) for x in acc_arr],
            "f1_per_seed":  [float(x) for x in f1_arr],
            "seeds": RANDOM_SEEDS,
        }

    json_path = os.path.join(results_dir, "visual_ablation_results.json")
    save_results_json(json_path, summary_vis)
    print(f"\nResults saved → {json_path}")

    # ── Print summary ────────────────────────────────────────────────────────
    print("\n" + "="*75)
    print("VISUAL ABLATION SUMMARY  (mean ± std over seeds: " + str(RANDOM_SEEDS) + ")")
    print("="*75)
    print(f"{'Experiment':<30} {'Acc (%)':<20} {'F1 (macro)':<20}")
    print("-"*75)
    for name, r in summary_vis.items():
        print(f"{name:<30} {r['acc_mean']*100:>7.2f} ± {r['acc_std']*100:<5.2f}  "
              f"{r['f1_mean']:>8.4f} ± {r['f1_std']:.4f}")


---
# Section 4 — Optimized Audiovisual Fusion

Isolates the best-performing signal from the visual ablation: Facial Action Units (FAU). Combines Audio (HuBERT) + FAU in two configurations:

- **Audio + Speaker FAU + Partner FAU** (3-stream dyadic model)
- **Audio + Speaker FAU only** (2-stream speaker-only model)

Both use `RANDOM_SEEDS` and save one `.pt` each. Results are merged into `audio_fau_fusion_results.json`.

## Dataset & MixUp

In [7]:
class DyadicAudioVisualDataset(Dataset):
    """
    Loads Audio, Self-Video, and Partner-Video.
    Slicing to isolate FAU happens in the training loop.
    """
    def __init__(self, df, audio_dict, self_video_dict, partner_video_dict):
        self.samples = []
        missing = 0
        for _, row in df.iterrows():
            s_id = row["sample_id"]
            if s_id in audio_dict and s_id in self_video_dict:
                # Handle missing partner videos (the 57 samples) by falling back to zeros
                vp = partner_video_dict.get(s_id, torch.zeros_like(self_video_dict[s_id]))
                self.samples.append({
                    "label": int(row["label"]),
                    "audio": audio_dict[s_id],
                    "video_self": self_video_dict[s_id],
                    "video_partner": vp,
                })
            else:
                missing += 1
        if missing > 0:
            print(f"  [Dataset] Skipped {missing} samples missing from dictionaries.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        return (
            item["audio"].clone().float(),
            item["video_self"].clone().float(),
            item["video_partner"].clone().float(),
            torch.tensor(item["label"], dtype=torch.long),
        )

def mixup_3stream(x_A, x_VS, x_VP, y, alpha: float = 0.2):
    batch_size = y.size(0)
    num_classes = 2

    if alpha <= 0:
        y_soft = torch.zeros(batch_size, num_classes, device=y.device).scatter_(1, y.unsqueeze(1), 1.0)
        return x_A, x_VS, x_VP, y_soft

    lam = np.random.beta(alpha, alpha)
    lam = max(lam, 1 - lam)
    idx = torch.randperm(batch_size, device=x_A.device)

    x_A  = lam * x_A  + (1 - lam) * x_A[idx]
    x_VS = lam * x_VS + (1 - lam) * x_VS[idx]
    x_VP = lam * x_VP + (1 - lam) * x_VP[idx]

    y_a = torch.zeros(batch_size, num_classes, device=y.device).scatter_(1, y.unsqueeze(1), 1.0)
    y_b = torch.zeros(batch_size, num_classes, device=y.device).scatter_(1, y[idx].unsqueeze(1), 1.0)
    y_soft = lam * y_a + (1 - lam) * y_b
    
    return x_A, x_VS, x_VP, y_soft


## Model: Audio_DyadicFAU_MulT (3-stream)

Intra-speaker: Audio ↔ Speaker FAU. Cross-speaker: Speaker FAU ↔ Partner FAU.

In [ ]:
"""
Tests the fusion of Acoustic Prosody (Audio) + Speaker Facial Action Units (VS_FAU) 
+ Partner Facial Action Units (VP_FAU) to capture the purest dyadic conflict signal.
"""

class Audio_DyadicFAU_MulT(nn.Module):
    """
    Audio + Speaker FAU + Partner FAU
    Intra-speaker: Audio <-> Speaker FAU
    Cross-speaker: Speaker FAU <-> Partner FAU
    """
    def __init__(self, audio_dim=768, fau_dim=24, shared_dim=64, num_heads=4):
        super().__init__()
        self.shared_dim = shared_dim
        
        # Projections
        self.proj_audio  = nn.Conv1d(audio_dim, shared_dim, kernel_size=1)
        self.proj_vs_fau = nn.Conv1d(fau_dim, shared_dim, kernel_size=1)
        self.proj_vp_fau = nn.Conv1d(fau_dim, shared_dim, kernel_size=1)
        
        self.pos_encoder = PositionalEncoding(d_model=shared_dim, dropout=0.2)
        
        # Intra-Speaker Blocks
        self.trans_A_vsFAU = CrossModalAttentionBlock(shared_dim, num_heads)
        self.trans_vsFAU_A = CrossModalAttentionBlock(shared_dim, num_heads)
        
        # Cross-Speaker Blocks (Dyadic Extension)
        self.trans_vsFAU_vpFAU = CrossModalAttentionBlock(shared_dim, num_heads)
        self.trans_vpFAU_vsFAU = CrossModalAttentionBlock(shared_dim, num_heads)
        
        # Pools & Gates
        self.pool_A      = AttentionPool(shared_dim)
        self.pool_vs_FAU = AttentionPool(shared_dim)
        self.pool_vp_FAU = AttentionPool(shared_dim)
        
        self.gate_A      = nn.Parameter(torch.ones(1))
        self.gate_vs_FAU = nn.Parameter(torch.ones(1))
        self.gate_vp_FAU = nn.Parameter(torch.ones(1))
        
        # Classifier (3 streams * 64 = 192 inputs)
        self.classifier = nn.Sequential(
            nn.Linear(shared_dim * 3, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(64, 2)
        )

    def forward(self, x_audio, x_vs_fau, x_vp_fau):
        # Encode
        proj_A   = self.pos_encoder(self.proj_audio(x_audio.transpose(1, 2)).transpose(1, 2))
        proj_vsF = self.pos_encoder(self.proj_vs_fau(x_vs_fau.transpose(1, 2)).transpose(1, 2))
        proj_vpF = self.pos_encoder(self.proj_vp_fau(x_vp_fau.transpose(1, 2)).transpose(1, 2))
        
        # Attention Routing
        out_A   = self.trans_A_vsFAU(proj_A, proj_vsF)
        
        out_vsF = self.trans_vsFAU_A(proj_vsF, proj_A)
        out_vsF = self.trans_vsFAU_vpFAU(out_vsF, proj_vpF) # Speaker attends to Partner reaction
        
        out_vpF = self.trans_vpFAU_vsFAU(proj_vpF, proj_vsF) # Partner reaction attends to Speaker
        
        # Pool
        p_A   = self.pool_A(out_A)
        p_vsF = self.pool_vs_FAU(out_vsF)
        p_vpF = self.pool_vp_FAU(out_vpF)
        
        # Gate & Concat
        fused = torch.cat([
            torch.sigmoid(self.gate_A) * p_A, 
            torch.sigmoid(self.gate_vs_FAU) * p_vsF,
            torch.sigmoid(self.gate_vp_FAU) * p_vpF
        ], dim=-1)
        
        return self.classifier(fused)


## Training Loop — 3-Stream

Trains `Audio_DyadicFAU_MulT` across all seeds, saves the single best `.pt`, and writes results to `audio_fau_fusion_results.json`.

In [ ]:
# TRAINING LOOP

def train_and_evaluate(
    model, train_loader, test_loader, device, train_df, 
    fau_slice, epochs=25, lr=1e-4, mixup_alpha=0.2, label_smooth=0.05
):
    experiment_name = "Audio_DyadicFAU"
    print(f"\n{'='*55}\nSTARTING EXPERIMENT: {experiment_name}\n{'='*55}\n")

    model = model.to(device)

    class_counts = train_df["label"].value_counts().sort_index().values
    weights = torch.tensor(1.0 / class_counts.astype(float), dtype=torch.float32)
    weights = (weights / weights.sum()).to(device)

    total_steps  = epochs * len(train_loader)
    warmup_steps = len(train_loader)
    optimizer    = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler    = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    train_losses, val_losses, val_accuracies = [], [], []
    best_val_loss  = float("inf")
    best_model_wts = copy.deepcopy(model.state_dict())
    best_epoch     = 1
    patience       = 10
    patience_ctr   = 0

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for batch_A, batch_VS, batch_VP, batch_y in train_loader:
            batch_A  = batch_A.to(device)
            batch_VS = batch_VS.to(device)
            batch_VP = batch_VP.to(device)
            batch_y  = batch_y.to(device)
            
            # SLICE THE VISUAL TENSORS (ONLY FAU)
            batch_vs_FAU = batch_VS[:, :, fau_slice[0]:fau_slice[1]]
            batch_vp_FAU = batch_VP[:, :, fau_slice[0]:fau_slice[1]]

            # MixUp
            batch_A, batch_vs_FAU, batch_vp_FAU, soft_y = mixup_3stream(
                batch_A, batch_vs_FAU, batch_vp_FAU, batch_y, alpha=mixup_alpha
            )

            # Label smoothing
            if label_smooth > 0:
                soft_y = (1 - label_smooth) * soft_y + label_smooth / soft_y.size(1)

            logits = model(batch_A, batch_vs_FAU, batch_vp_FAU)

            log_probs = F.log_softmax(logits, dim=-1)
            weighted_log = log_probs * weights.unsqueeze(0)
            loss = -(soft_y * weighted_log).sum(dim=-1).mean()

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            running_loss += loss.item()

        epoch_train_loss = running_loss / len(train_loader)
        train_losses.append(epoch_train_loss)

        # ---- Validation ----
        model.eval()
        running_val_loss = 0.0
        val_preds, val_targets = [], []

        with torch.no_grad():
            for batch_A, batch_VS, batch_VP, batch_y in test_loader:
                batch_A  = batch_A.to(device)
                batch_VS = batch_VS.to(device)
                batch_VP = batch_VP.to(device)
                batch_y  = batch_y.to(device)
                
                batch_vs_FAU = batch_VS[:, :, fau_slice[0]:fau_slice[1]]
                batch_vp_FAU = batch_VP[:, :, fau_slice[0]:fau_slice[1]]

                logits = model(batch_A, batch_vs_FAU, batch_vp_FAU)

                y_onehot = torch.zeros_like(logits).scatter_(1, batch_y.unsqueeze(1), 1.0)
                log_probs = F.log_softmax(logits, dim=-1)
                val_loss  = -(y_onehot * (log_probs * weights.unsqueeze(0))).sum(dim=-1).mean()
                running_val_loss += val_loss.item()

                _, predicted = torch.max(logits, 1)
                val_preds.extend(predicted.cpu().numpy())
                val_targets.extend(batch_y.cpu().numpy())

        epoch_val_loss = running_val_loss / len(test_loader)
        epoch_val_acc  = accuracy_score(val_targets, val_preds)
        val_losses.append(epoch_val_loss)
        val_accuracies.append(epoch_val_acc)

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch [{epoch+1:02d}/{epochs}] | Train Loss: {epoch_train_loss:.4f} | "
                  f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc*100:.2f}% | "
                  f"LR: {optimizer.param_groups[0]['lr']:.2e}")

        if epoch_val_loss < best_val_loss:
            best_val_loss  = epoch_val_loss
            best_model_wts = copy.deepcopy(model.state_dict())
            best_epoch     = epoch + 1
            patience_ctr   = 0
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print(f"\n  Early stopping at epoch {epoch+1} (no improvement for {patience} epochs).")
                break

    print(f"\nRestoring best weights from Epoch {best_epoch} (Val Loss: {best_val_loss:.4f})")
    model.load_state_dict(best_model_wts)

    # ---- Final Evaluation ----
    model.eval()
    final_preds, final_targets = [], []
    with torch.no_grad():
        for batch_A, batch_VS, batch_VP, batch_y in test_loader:
            batch_vs_FAU = batch_VS[:, :, fau_slice[0]:fau_slice[1]].to(device)
            batch_vp_FAU = batch_VP[:, :, fau_slice[0]:fau_slice[1]].to(device)
            
            logits = model(batch_A.to(device), batch_vs_FAU, batch_vp_FAU)
            
            _, predicted = torch.max(logits, 1)
            final_preds.extend(predicted.cpu().numpy())
            final_targets.extend(batch_y.numpy())

    best_acc = accuracy_score(final_targets, final_preds)
    f1       = f1_score(final_targets, final_preds, average="macro")

    print("\n" + "="*50)
    print(f"FINAL RESULTS: {experiment_name}")
    print("="*50)
    print(f"Accuracy : {best_acc*100:.2f}%")
    print(f"F1 (macro): {f1:.4f}")
    print("-"*50)
    print(classification_report(final_targets, final_preds, target_names=["Calm (0)", "Conflict (1)"]))

    gA  = torch.sigmoid(model.gate_A).item()
    gVS = torch.sigmoid(model.gate_vs_FAU).item()
    gVP = torch.sigmoid(model.gate_vp_FAU).item()
    print(f"Learned gates — Audio: {gA:.3f} | Speaker FAU: {gVS:.3f} | Partner FAU: {gVP:.3f}")

    # ---- Plotting ----
    plot_dir = PATHS["plots"]
    os.makedirs(plot_dir, exist_ok=True)
    epochs_run = len(train_losses)

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(range(1, epochs_run + 1), train_losses, label="Train Loss", color="steelblue", linewidth=2)
    plt.plot(range(1, epochs_run + 1), val_losses, label="Val Loss", color="tomato", linewidth=2, linestyle="--")
    plt.axvline(x=best_epoch, color="green", linestyle=":", label=f"Best Epoch {best_epoch}")
    plt.title(f"Loss: {experiment_name}", fontsize=12)
    plt.legend(); plt.grid(True, alpha=0.3)

    plt.subplot(1, 2, 2)
    plt.plot(range(1, epochs_run + 1), [a * 100 for a in val_accuracies], label="Val Accuracy", color="mediumseagreen", linewidth=2)
    plt.axvline(x=best_epoch, color="red", linestyle=":", label=f"Best Epoch {best_epoch}")
    plt.title(f"Accuracy: {experiment_name}", fontsize=12)
    plt.legend(); plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(plot_dir, f"{experiment_name}_curves.png"), dpi=200)
    plt.close()

    cm = confusion_matrix(final_targets, final_preds)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Predicted Calm", "Predicted Conflict"],
                yticklabels=["Actual Calm", "Actual Conflict"], annot_kws={"size": 14})
    plt.title(f"Confusion Matrix: {experiment_name}", fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(plot_dir, f"{experiment_name}_cm.png"), dpi=200)
    plt.close()

    # Save model weights
    # (weights saved externally by the multi-seed runner)

if __name__ == "__main__":

    # ── Load data once ───────────────────────────────────────────────────────
    with open(EMBEDDINGS["video_feat_dims"]) as f:
        fau_slice = json.load(f)["fau_slice"]

    df = pd.read_csv(PATHS["csv"])

    print("Loading embeddings...")
    audio_dict         = load_embedding("audio")
    self_video_dict    = load_embedding("video_self")
    partner_video_dict = load_embedding("video_partner")

    train_df, test_df = group_split(df, test_size=0.20, seed=42)

    train_dataset = DyadicAudioVisualDataset(train_df, audio_dict,
                                             self_video_dict, partner_video_dict)
    test_dataset  = DyadicAudioVisualDataset(test_df,  audio_dict,
                                             self_video_dict, partner_video_dict)

    AUDIO_DIM = next(iter(audio_dict.values())).shape[-1]
    FAU_DIM   = fau_slice[1] - fau_slice[0]

    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,
                              num_workers=8, pin_memory=True)
    test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False,
                              num_workers=8, pin_memory=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    save_dir    = PATHS["models"]
    results_dir = PATHS["results"]

    acc_runs, f1_runs = [], []
    best_f1_global = -1.0
    best_state_dict = None
    best_seed = None

    for seed in RANDOM_SEEDS:
        print(f"\n{'#'*65}")
        print(f"  Audio + Speaker FAU + Partner FAU — SEED {seed}")
        print(f"{'#'*65}")
        set_seed(seed)

        model = Audio_DyadicFAU_MulT(
            audio_dim=AUDIO_DIM, fau_dim=FAU_DIM, shared_dim=64, num_heads=4
        )
        acc, f1 = train_and_evaluate(
            model, train_loader, test_loader, device, train_df,
            fau_slice, epochs=25, lr=1e-4
        )
        acc_runs.append(acc)
        f1_runs.append(f1)

        if f1 >= best_f1_global:
            best_f1_global = f1
            best_state_dict = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            best_seed = seed

    # ── Save single best .pt ─────────────────────────────────────────────────
    save_path = os.path.join(save_dir, "Audio_DyadicFAU_best_weights.pt")
    torch.save(best_state_dict, save_path)
    print(f"\nSaved best Audio_DyadicFAU weights "
          f"(seed={best_seed}, F1={best_f1_global:.4f}) -> {save_path}")

    # ── Aggregate & save JSON ─────────────────────────────────────────────────
    summary_3stream = {"Audio_DyadicFAU": aggregate_seeds(acc_runs, f1_runs)}

    json_path = os.path.join(results_dir, "audio_fau_fusion_results.json")
    save_results_json(json_path, summary_3stream)
    print(f"Results saved → {json_path}")

    r = summary_3stream["Audio_DyadicFAU"]
    print(f"\nAudio + Speaker FAU + Partner FAU: "
          f"Acc = {r['acc_mean']*100:.2f} ± {r['acc_std']*100:.2f}%  |  "
          f"F1 = {r['f1_mean']:.4f} ± {r['f1_std']:.4f}")
    print(f"(Seeds: {RANDOM_SEEDS})")


## Model: Audio_FAU_MulT (2-stream)

Speaker-only baseline: Audio ↔ Speaker FAU, no partner stream.

In [ ]:
# 2-STREAM ARCHITECTURE

class Audio_FAU_MulT(nn.Module):
    def __init__(self, audio_dim=768, fau_dim=24, shared_dim=64, num_heads=4):
        super().__init__()
        self.shared_dim = shared_dim
        
        # Projections
        self.proj_audio = nn.Conv1d(audio_dim, shared_dim, kernel_size=1)
        self.proj_fau   = nn.Conv1d(fau_dim, shared_dim, kernel_size=1)
        
        self.pos_encoder = PositionalEncoding(d_model=shared_dim, dropout=0.2)
        
        # Cross Attention Blocks
        self.trans_A_FAU = CrossModalAttentionBlock(shared_dim, num_heads)
        self.trans_FAU_A = CrossModalAttentionBlock(shared_dim, num_heads)
        
        # Pools & Gates
        self.pool_A   = AttentionPool(shared_dim)
        self.pool_FAU = AttentionPool(shared_dim)
        
        self.gate_A   = nn.Parameter(torch.ones(1))
        self.gate_FAU = nn.Parameter(torch.ones(1))
        
        # Classifier (2 streams * 64 = 128 inputs)
        self.classifier = nn.Sequential(
            nn.Linear(shared_dim * 2, 64),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(64, 32),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(32, 2)
        )

    def forward(self, x_audio, x_fau):
        # Encode
        proj_A   = self.pos_encoder(self.proj_audio(x_audio.transpose(1, 2)).transpose(1, 2))
        proj_FAU = self.pos_encoder(self.proj_fau(x_fau.transpose(1, 2)).transpose(1, 2))
        
        # Attention
        out_A   = self.trans_A_FAU(proj_A, proj_FAU)
        out_FAU = self.trans_FAU_A(proj_FAU, proj_A)
        
        # Pool
        p_A   = self.pool_A(out_A)
        p_FAU = self.pool_FAU(out_FAU)
        
        # Gate & Concat
        fused = torch.cat([
            torch.sigmoid(self.gate_A) * p_A, 
            torch.sigmoid(self.gate_FAU) * p_FAU
        ], dim=-1)
        
        return self.classifier(fused)


## Dataset & MixUp -- 2 Stream 

In [ ]:
class AudioVisualDataset(Dataset):
    """
    Loads only Audio and Self-Video for the 2-stream baseline.
    Slicing to isolate FAU happens in the training loop.
    """
    def __init__(self, df, audio_dict, self_video_dict):
        self.samples = []
        missing = 0
        for _, row in df.iterrows():
            s_id = row["sample_id"]
            if s_id in audio_dict and s_id in self_video_dict:
                self.samples.append({
                    "label": int(row["label"]),
                    "audio": audio_dict[s_id],
                    "video_self": self_video_dict[s_id],
                })
            else:
                missing += 1
        if missing > 0:
            print(f"  [Dataset] Skipped {missing} samples missing from dictionaries.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        return (
            item["audio"].clone().float(),
            item["video_self"].clone().float(),
            torch.tensor(item["label"], dtype=torch.long),
        )


def mixup_2stream(x_A, x_VS, y, alpha: float = 0.2):
    """MixUp for 2 streams (Audio + Speaker Video)."""
    batch_size = y.size(0)
    num_classes = 2

    if alpha <= 0:
        y_soft = torch.zeros(batch_size, num_classes, device=y.device).scatter_(1, y.unsqueeze(1), 1.0)
        return x_A, x_VS, y_soft

    lam = np.random.beta(alpha, alpha)
    lam = max(lam, 1 - lam)
    idx = torch.randperm(batch_size, device=x_A.device)

    x_A  = lam * x_A  + (1 - lam) * x_A[idx]
    x_VS = lam * x_VS + (1 - lam) * x_VS[idx]

    y_a = torch.zeros(batch_size, num_classes, device=y.device).scatter_(1, y.unsqueeze(1), 1.0)
    y_b = torch.zeros(batch_size, num_classes, device=y.device).scatter_(1, y[idx].unsqueeze(1), 1.0)
    y_soft = lam * y_a + (1 - lam) * y_b
    
    return x_A, x_VS, y_soft

## Training Loop — 2-Stream

Trains `Audio_FAU_MulT` across all seeds, saves the single best `.pt`, and merges results into `audio_fau_fusion_results.json`. Prints a combined summary table once both models have been run.

In [ ]:
# TRAINING LOOP

def train_and_evaluate_2stream(
    model, train_loader, test_loader, device, train_df, 
    fau_slice, epochs=25, lr=1e-4, mixup_alpha=0.2, label_smooth=0.05
):
    experiment_name = "Audio_FAU_Only"
    print(f"\n{'='*55}\nSTARTING EXPERIMENT: {experiment_name}\n{'='*55}\n")

    model = model.to(device)

    class_counts = train_df["label"].value_counts().sort_index().values
    weights = torch.tensor(1.0 / class_counts.astype(float), dtype=torch.float32)
    weights = (weights / weights.sum()).to(device)

    total_steps  = epochs * len(train_loader)
    warmup_steps = len(train_loader)
    optimizer    = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler    = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    train_losses, val_losses, val_accuracies = [], [], []
    best_val_loss  = float("inf")
    best_model_wts = copy.deepcopy(model.state_dict())
    best_epoch     = 1
    patience       = 10
    patience_ctr   = 0

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for batch_A, batch_VS, batch_y in train_loader:
            batch_A  = batch_A.to(device)
            batch_VS = batch_VS.to(device)
            batch_y  = batch_y.to(device)
            
            # SLICE THE VISUAL TENSOR (ONLY FAU)
            batch_FAU = batch_VS[:, :, fau_slice[0]:fau_slice[1]]

            # MixUp
            batch_A, batch_FAU, soft_y = mixup_2stream(
                batch_A, batch_FAU, batch_y, alpha=mixup_alpha
            )

            # Label smoothing
            if label_smooth > 0:
                soft_y = (1 - label_smooth) * soft_y + label_smooth / soft_y.size(1)

            logits = model(batch_A, batch_FAU)

            log_probs = F.log_softmax(logits, dim=-1)
            weighted_log = log_probs * weights.unsqueeze(0)
            loss = -(soft_y * weighted_log).sum(dim=-1).mean()

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            running_loss += loss.item()

        epoch_train_loss = running_loss / len(train_loader)
        train_losses.append(epoch_train_loss)

        # ---- Validation ----
        model.eval()
        running_val_loss = 0.0
        val_preds, val_targets = [], []

        with torch.no_grad():
            for batch_A, batch_VS, batch_y in test_loader:
                batch_A  = batch_A.to(device)
                batch_VS = batch_VS.to(device)
                batch_y  = batch_y.to(device)
                
                batch_FAU = batch_VS[:, :, fau_slice[0]:fau_slice[1]]

                logits = model(batch_A, batch_FAU)

                y_onehot = torch.zeros_like(logits).scatter_(1, batch_y.unsqueeze(1), 1.0)
                log_probs = F.log_softmax(logits, dim=-1)
                val_loss  = -(y_onehot * (log_probs * weights.unsqueeze(0))).sum(dim=-1).mean()
                running_val_loss += val_loss.item()

                _, predicted = torch.max(logits, 1)
                val_preds.extend(predicted.cpu().numpy())
                val_targets.extend(batch_y.cpu().numpy())

        epoch_val_loss = running_val_loss / len(test_loader)
        epoch_val_acc  = accuracy_score(val_targets, val_preds)
        val_losses.append(epoch_val_loss)
        val_accuracies.append(epoch_val_acc)

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch [{epoch+1:02d}/{epochs}] | Train Loss: {epoch_train_loss:.4f} | "
                  f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc*100:.2f}% | "
                  f"LR: {optimizer.param_groups[0]['lr']:.2e}")

        if epoch_val_loss < best_val_loss:
            best_val_loss  = epoch_val_loss
            best_model_wts = copy.deepcopy(model.state_dict())
            best_epoch     = epoch + 1
            patience_ctr   = 0
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print(f"\n  Early stopping at epoch {epoch+1} (no improvement for {patience} epochs).")
                break

    print(f"\nRestoring best weights from Epoch {best_epoch} (Val Loss: {best_val_loss:.4f})")
    model.load_state_dict(best_model_wts)

    # ---- Final Evaluation ----
    model.eval()
    final_preds, final_targets = [], []
    with torch.no_grad():
        for batch_A, batch_VS, batch_y in test_loader:
            batch_FAU = batch_VS[:, :, fau_slice[0]:fau_slice[1]].to(device)
            logits = model(batch_A.to(device), batch_FAU)
            
            _, predicted = torch.max(logits, 1)
            final_preds.extend(predicted.cpu().numpy())
            final_targets.extend(batch_y.numpy())

    best_acc = accuracy_score(final_targets, final_preds)
    f1       = f1_score(final_targets, final_preds, average="macro")

    print("\n" + "="*50)
    print(f"FINAL RESULTS: {experiment_name}")
    print("="*50)
    print(f"Accuracy : {best_acc*100:.2f}%")
    print(f"F1 (macro): {f1:.4f}")
    print("-"*50)
    print(classification_report(final_targets, final_preds, target_names=["Calm (0)", "Conflict (1)"]))

    gA  = torch.sigmoid(model.gate_A).item()
    gFAU = torch.sigmoid(model.gate_FAU).item()
    print(f"Learned gates — Audio: {gA:.3f} | FAU: {gFAU:.3f}")

    # ---- Plotting ----
    plot_dir = PATHS["plots"]
    os.makedirs(plot_dir, exist_ok=True)
    epochs_run = len(train_losses)

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(range(1, epochs_run + 1), train_losses, label="Train Loss", color="steelblue", linewidth=2)
    plt.plot(range(1, epochs_run + 1), val_losses, label="Val Loss", color="tomato", linewidth=2, linestyle="--")
    plt.axvline(x=best_epoch, color="green", linestyle=":", label=f"Best Epoch {best_epoch}")
    plt.title(f"Loss: {experiment_name}", fontsize=12)
    plt.legend(); plt.grid(True, alpha=0.3)

    plt.subplot(1, 2, 2)
    plt.plot(range(1, epochs_run + 1), [a * 100 for a in val_accuracies], label="Val Accuracy", color="mediumseagreen", linewidth=2)
    plt.axvline(x=best_epoch, color="red", linestyle=":", label=f"Best Epoch {best_epoch}")
    plt.title(f"Accuracy: {experiment_name}", fontsize=12)
    plt.legend(); plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(plot_dir, f"{experiment_name}_curves.png"), dpi=200)
    plt.close()

    # Save model weights
    # (weights saved externally by the multi-seed runner)

if __name__ == "__main__":

    # ── Load data once ───────────────────────────────────────────────────────
    with open(EMBEDDINGS["video_feat_dims"]) as f:
        fau_slice = json.load(f)["fau_slice"]

    df = pd.read_csv(PATHS["csv"])

    print("Loading embeddings...")
    audio_dict      = load_embedding("audio")
    self_video_dict = load_embedding("video_self")

    train_df, test_df = group_split(df, test_size=0.20, seed=42)

    train_dataset = AudioVisualDataset(train_df, audio_dict, self_video_dict)
    test_dataset  = AudioVisualDataset(test_df,  audio_dict, self_video_dict)

    AUDIO_DIM = next(iter(audio_dict.values())).shape[-1]
    FAU_DIM   = fau_slice[1] - fau_slice[0]

    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,
                              num_workers=8, pin_memory=True)
    test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False,
                              num_workers=8, pin_memory=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    save_dir    = PATHS["models"]
    results_dir = PATHS["results"]

    acc_runs, f1_runs = [], []
    best_f1_global = -1.0
    best_state_dict = None
    best_seed = None

    for seed in RANDOM_SEEDS:
        print(f"\n{'#'*65}")
        print(f"  Audio + Speaker FAU Only — SEED {seed}")
        print(f"{'#'*65}")
        set_seed(seed)

        model = Audio_FAU_MulT(
            audio_dim=AUDIO_DIM, fau_dim=FAU_DIM, shared_dim=64, num_heads=4
        )
        acc, f1 = train_and_evaluate_2stream(
            model, train_loader, test_loader, device, train_df,
            fau_slice, epochs=25, lr=1e-4
        )
        acc_runs.append(acc)
        f1_runs.append(f1)

        if f1 >= best_f1_global:
            best_f1_global = f1
            best_state_dict = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            best_seed = seed

    # ── Save single best .pt ─────────────────────────────────────────────────
    save_path = os.path.join(save_dir, "Audio_FAU_Only_best_weights.pt")
    torch.save(best_state_dict, save_path)
    print(f"\nSaved best Audio_FAU_Only weights "
          f"(seed={best_seed}, F1={best_f1_global:.4f}) -> {save_path}")

    # ── Aggregate & save JSON ─────────────────────────────────────────────────
    result_entry = {"Audio_FAU_Only": aggregate_seeds(acc_runs, f1_runs)}

    json_path = os.path.join(results_dir, "audio_fau_fusion_results.json")
    save_results_json(json_path, result_entry)
    print(f"Results saved → {json_path}")

    r = result_entry["Audio_FAU_Only"]
    print(f"\nAudio + Speaker FAU Only: "
          f"Acc = {r['acc_mean']*100:.2f} ± {r['acc_std']*100:.2f}%  |  "
          f"F1 = {r['f1_mean']:.4f} ± {r['f1_std']:.4f}")
    print(f"(Seeds: {RANDOM_SEEDS})")

    # ── Combined summary (shared helper) ─────────────────────────────────
    opt_json = os.path.join(results_dir, "audio_fau_fusion_results.json")
    if os.path.exists(opt_json):
        with open(opt_json) as f:
            both = json.load(f)
        if "Audio_DyadicFAU" in both and "Audio_FAU_Only" in both:
            print_summary_table("OPTIMIZED AUDIOVISUAL FUSION SUMMARY", both)


## Training Loop — 5-Stream

Trains `Audio_FAU_Body_MulT` across all seeds, saves the single best `.pt`, and merges results into `audio_fau_body_fusion_results.json`. Prints a combined summary table once both models have been run.

In [ ]:
def train_eval_fusion_5stream(model, train_loader, test_loader, device, train_df, fau_slice, body_slice, exp_name):
    """
    Training loop for the optimized 5-stream dyadic model.
    Dynamically slices the visual tensors to isolate the FAU and Body features.
    """
    model = model.to(device)
    
    # ---- 1. Class Weights ----
    class_counts = train_df["label"].value_counts().sort_index().values
    weights = torch.tensor(1.0 / class_counts.astype(float), dtype=torch.float32).to(device)
    weights = weights / weights.sum()
    
    # ---- 2. Optimizer & Scheduler ----
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-3)
    total_steps = 25 * len(train_loader)
    warmup_steps = len(train_loader)
    scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    
    # ---- 3. Tracking Metrics ----
    t_loss, v_loss, v_acc = [], [], []
    best_loss = float("inf")
    best_wts = copy.deepcopy(model.state_dict())
    best_ep = 1
    patience_ctr = 0
    
    for epoch in range(25):
        # ─── TRAINING ──────────────────────────────────────
        model.train()
        run_loss = 0.0
        
        for batch in train_loader:
            bA, bVS, bVP, by = [t.to(device) for t in batch]
            
            # Dynamic Slicing: Extract the 24-d FAU and 63-d Body blocks
            bVS_FAU = bVS[:, :, fau_slice[0]:fau_slice[1]]
            bVS_Bod = bVS[:, :, body_slice[0]:body_slice[1]]
            bVP_FAU = bVP[:, :, fau_slice[0]:fau_slice[1]]
            bVP_Bod = bVP[:, :, body_slice[0]:body_slice[1]]
            
            mixed, soft_y = mixup_fusion([bA, bVS_FAU, bVS_Bod, bVP_FAU, bVP_Bod], by, 0.2)
            logits = model(*mixed)
                
            # Label Smoothing
            soft_y = 0.95 * soft_y + 0.05 / 2
            
            # Class-weighted soft cross-entropy
            log_probs = F.log_softmax(logits, dim=-1)
            weighted_log = log_probs * weights.unsqueeze(0)
            loss = -(soft_y * weighted_log).sum(dim=-1).mean()
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            run_loss += loss.item()
            
        t_loss.append(run_loss / len(train_loader))
        
        # ─── VALIDATION ────────────────────────────────────
        model.eval()
        run_v_loss = 0.0
        preds, targets = [], []
        
        with torch.no_grad():
            for batch in test_loader:
                bA, bVS, bVP, by = [t.to(device) for t in batch]
                
                bVS_FAU = bVS[:, :, fau_slice[0]:fau_slice[1]]
                bVS_Bod = bVS[:, :, body_slice[0]:body_slice[1]]
                bVP_FAU = bVP[:, :, fau_slice[0]:fau_slice[1]]
                bVP_Bod = bVP[:, :, body_slice[0]:body_slice[1]]
                
                logits = model(bA, bVS_FAU, bVS_Bod, bVP_FAU, bVP_Bod)
                    
                # Hard-label cross entropy for validation monitoring
                y_onehot = torch.zeros_like(logits).scatter_(1, by.unsqueeze(1), 1.0)
                log_probs = F.log_softmax(logits, dim=-1)
                val_loss = -(y_onehot * (log_probs * weights.unsqueeze(0))).sum(dim=-1).mean()
                
                run_v_loss += val_loss.item()
                preds.extend(torch.max(logits, 1)[1].cpu().numpy())
                targets.extend(by.cpu().numpy())
                
        v_loss.append(run_v_loss / len(test_loader))
        v_acc.append(accuracy_score(targets, preds))
        
        # ─── EARLY STOPPING ────────────────────────────────
        if v_loss[-1] < best_loss:
            best_loss = v_loss[-1]
            best_wts = copy.deepcopy(model.state_dict())
            best_ep = epoch + 1
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= 10:
                break

    # ---- 4. Final Evaluation with Best Weights ----
    model.load_state_dict(best_wts)
    model.eval()
    f_preds, f_targets = [], []
    
    with torch.no_grad():
        for batch in test_loader:
            bA, bVS, bVP, by = [t.to(device) for t in batch]
            
            bVS_FAU = bVS[:, :, fau_slice[0]:fau_slice[1]]
            bVS_Bod = bVS[:, :, body_slice[0]:body_slice[1]]
            bVP_FAU = bVP[:, :, fau_slice[0]:fau_slice[1]]
            bVP_Bod = bVP[:, :, body_slice[0]:body_slice[1]]
            
            logits = model(bA, bVS_FAU, bVS_Bod, bVP_FAU, bVP_Bod)
            
            f_preds.extend(torch.max(logits, 1)[1].cpu().numpy())
            f_targets.extend(by.cpu().numpy())
            
    acc = accuracy_score(f_targets, f_preds)
    f1 = f1_score(f_targets, f_preds, average="macro")
    
    return acc, f1, best_wts

# Section 5 — 5-Stream Audiovisual Fusion

Extends Section 4 by adding the body pose stream alongside FAU. Five streams total:
Audio (HuBERT) + Speaker FAU + Speaker Body + Partner FAU + Partner Body.
Runs across `RANDOM_SEEDS`, saves one best-seed `.pt`, and writes results to
`fusion_Audio_DyadicFAUBody.json`.

## Dataset & MixUp

Same dataset and MixUp as Section 4 — redefined here so Section 5 can run standalone without requiring Section 4 cells to be executed first.

In [7]:
class DyadicAudioVisualDataset(Dataset):
    """
    Loads Audio, Self-Video, and Partner-Video.
    Slicing to isolate FAU / Body sub-streams happens in the training loop.
    """
    def __init__(self, df, audio_dict, self_video_dict, partner_video_dict):
        self.samples = []
        missing = 0
        for _, row in df.iterrows():
            s_id = row["sample_id"]
            if s_id in audio_dict and s_id in self_video_dict:
                vp = partner_video_dict.get(s_id, torch.zeros_like(self_video_dict[s_id]))
                self.samples.append({
                    "label": int(row["label"]),
                    "audio": audio_dict[s_id],
                    "video_self": self_video_dict[s_id],
                    "video_partner": vp,
                })
            else:
                missing += 1
        if missing:
            print(f"  [{self.__class__.__name__}] Skipped {missing} samples with missing embeddings.")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        i = self.samples[idx]
        return (
            i["audio"].clone().float(),
            i["video_self"].clone().float(),
            i["video_partner"].clone().float(),
            torch.tensor(i["label"], dtype=torch.long),
        )


def mixup_fusion(streams, y, alpha=0.2):
    """MixUp for an arbitrary list of stream tensors with soft labels."""
    B, NC = y.size(0), 2
    if alpha <= 0:
        return streams, torch.zeros(B, NC, device=y.device).scatter_(1, y.unsqueeze(1), 1.0)
    lam = max(np.random.beta(alpha, alpha), 1 - np.random.beta(alpha, alpha))
    idx = torch.randperm(B, device=streams[0].device)
    mixed = [lam * s + (1 - lam) * s[idx] for s in streams]
    y_a = torch.zeros(B, NC, device=y.device).scatter_(1, y.unsqueeze(1), 1.0)
    y_b = torch.zeros(B, NC, device=y.device).scatter_(1, y[idx].unsqueeze(1), 1.0)
    return mixed, lam * y_a + (1 - lam) * y_b

## Model: Audio_DyadicFAUBody_MulT (5-stream)

Intra-speaker: Audio ↔ Speaker FAU, Audio ↔ Speaker Body, Speaker FAU ↔ Speaker Body. Cross-speaker: Speaker FAU ↔ Partner FAU, Speaker Body ↔ Partner Body.

In [8]:
"""
Audio + Speaker FAU + Speaker Body + Partner FAU + Partner Body.
Intra-speaker cross-modal attention between Audio, FAU, and Body.
Cross-speaker dyadic attention between matching sub-streams.
"""

class Audio_DyadicFAUBody_MulT(nn.Module):
    def __init__(self, audio_dim=768, fau_dim=24, body_dim=63, shared_dim=64, num_heads=4):
        super().__init__()
        self.proj_audio = nn.Conv1d(audio_dim, shared_dim, 1)
        self.proj_vs_fau, self.proj_vp_fau = nn.Conv1d(fau_dim, shared_dim, 1), nn.Conv1d(fau_dim, shared_dim, 1)
        self.proj_vs_bod, self.proj_vp_bod = nn.Conv1d(body_dim, shared_dim, 1), nn.Conv1d(body_dim, shared_dim, 1)
        self.pos_encoder = PositionalEncoding(shared_dim, 0.2)
        # Intra-speaker cross-modal attention
        self.trans_A_vsFAU, self.trans_A_vsBod = CrossModalAttentionBlock(shared_dim, num_heads), CrossModalAttentionBlock(shared_dim, num_heads)
        self.trans_vsFAU_A, self.trans_vsFAU_Bod = CrossModalAttentionBlock(shared_dim, num_heads), CrossModalAttentionBlock(shared_dim, num_heads)
        self.trans_vsBod_A, self.trans_vsBod_FAU = CrossModalAttentionBlock(shared_dim, num_heads), CrossModalAttentionBlock(shared_dim, num_heads)
        # Cross-speaker dyadic attention
        self.trans_vsFAU_vpFAU, self.trans_vpFAU_vsFAU = CrossModalAttentionBlock(shared_dim, num_heads), CrossModalAttentionBlock(shared_dim, num_heads)
        self.trans_vsBod_vpBod, self.trans_vpBod_vsBod = CrossModalAttentionBlock(shared_dim, num_heads), CrossModalAttentionBlock(shared_dim, num_heads)
        self.pool_A, self.pool_vs_FAU, self.pool_vs_Bod = AttentionPool(shared_dim), AttentionPool(shared_dim), AttentionPool(shared_dim)
        self.pool_vp_FAU, self.pool_vp_Bod = AttentionPool(shared_dim), AttentionPool(shared_dim)
        self.gate_A, self.gate_vs_FAU, self.gate_vs_Bod = nn.Parameter(torch.ones(1)), nn.Parameter(torch.ones(1)), nn.Parameter(torch.ones(1))
        self.gate_vp_FAU, self.gate_vp_Bod = nn.Parameter(torch.ones(1)), nn.Parameter(torch.ones(1))
        self.classifier = nn.Sequential(
            nn.Linear(shared_dim * 5, 128), nn.BatchNorm1d(128), nn.GELU(), nn.Dropout(0.4),
            nn.Linear(128, 64), nn.GELU(), nn.Dropout(0.3), nn.Linear(64, 2)
        )

    def forward(self, x_audio, x_vs_fau, x_vs_bod, x_vp_fau, x_vp_bod):
        def _enc(x, proj): return self.pos_encoder(proj(x.transpose(1, 2)).transpose(1, 2))
        pA    = _enc(x_audio,  self.proj_audio)
        pVSF  = _enc(x_vs_fau, self.proj_vs_fau)
        pVSB  = _enc(x_vs_bod, self.proj_vs_bod)
        pVPF  = _enc(x_vp_fau, self.proj_vp_fau)
        pVPB  = _enc(x_vp_bod, self.proj_vp_bod)
        oA   = self.trans_A_vsBod(self.trans_A_vsFAU(pA, pVSF), pVSB)
        oVSF = self.trans_vsFAU_vpFAU(self.trans_vsFAU_Bod(self.trans_vsFAU_A(pVSF, pA), pVSB), pVPF)
        oVSB = self.trans_vsBod_vpBod(self.trans_vsBod_FAU(self.trans_vsBod_A(pVSB, pA), pVSF), pVPB)
        oVPF = self.trans_vpFAU_vsFAU(pVPF, pVSF)
        oVPB = self.trans_vpBod_vsBod(pVPB, pVSB)
        fused = torch.cat([
            torch.sigmoid(self.gate_A)      * self.pool_A(oA),
            torch.sigmoid(self.gate_vs_FAU) * self.pool_vs_FAU(oVSF),
            torch.sigmoid(self.gate_vs_Bod) * self.pool_vs_Bod(oVSB),
            torch.sigmoid(self.gate_vp_FAU) * self.pool_vp_FAU(oVPF),
            torch.sigmoid(self.gate_vp_Bod) * self.pool_vp_Bod(oVPB),
        ], dim=-1)
        return self.classifier(fused)

## Training Loop — 5-Stream

Trains `Audio_FAU_Body_MulT` across all seeds, saves the single best `.pt`, and merges results into `audio_fau_body_fusion_results.json`. Prints a combined summary table once both models have been run.

In [9]:
def train_eval_fusion_5stream(model, train_loader, test_loader, device, train_df, fau_slice, body_slice, exp_name):
    """
    Training loop for the optimized 5-stream dyadic model.
    Dynamically slices the visual tensors to isolate the FAU and Body features.
    """
    model = model.to(device)
    
    # ---- 1. Class Weights ----
    class_counts = train_df["label"].value_counts().sort_index().values
    weights = torch.tensor(1.0 / class_counts.astype(float), dtype=torch.float32).to(device)
    weights = weights / weights.sum()
    
    # ---- 2. Optimizer & Scheduler ----
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-3)
    total_steps = 25 * len(train_loader)
    warmup_steps = len(train_loader)
    scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    
    # ---- 3. Tracking Metrics ----
    t_loss, v_loss, v_acc = [], [], []
    best_loss = float("inf")
    best_wts = copy.deepcopy(model.state_dict())
    best_ep = 1
    patience_ctr = 0
    
    for epoch in range(25):
        # ─── TRAINING ──────────────────────────────────────
        model.train()
        run_loss = 0.0
        
        for batch in train_loader:
            bA, bVS, bVP, by = [t.to(device) for t in batch]
            
            # Dynamic Slicing: Extract the 24-d FAU and 63-d Body blocks
            bVS_FAU = bVS[:, :, fau_slice[0]:fau_slice[1]]
            bVS_Bod = bVS[:, :, body_slice[0]:body_slice[1]]
            bVP_FAU = bVP[:, :, fau_slice[0]:fau_slice[1]]
            bVP_Bod = bVP[:, :, body_slice[0]:body_slice[1]]
            
            mixed, soft_y = mixup_fusion([bA, bVS_FAU, bVS_Bod, bVP_FAU, bVP_Bod], by, 0.2)
            logits = model(*mixed)
                
            # Label Smoothing
            soft_y = 0.95 * soft_y + 0.05 / 2
            
            # Class-weighted soft cross-entropy
            log_probs = F.log_softmax(logits, dim=-1)
            weighted_log = log_probs * weights.unsqueeze(0)
            loss = -(soft_y * weighted_log).sum(dim=-1).mean()
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            run_loss += loss.item()
            
        t_loss.append(run_loss / len(train_loader))
        
        # ─── VALIDATION ────────────────────────────────────
        model.eval()
        run_v_loss = 0.0
        preds, targets = [], []
        
        with torch.no_grad():
            for batch in test_loader:
                bA, bVS, bVP, by = [t.to(device) for t in batch]
                
                bVS_FAU = bVS[:, :, fau_slice[0]:fau_slice[1]]
                bVS_Bod = bVS[:, :, body_slice[0]:body_slice[1]]
                bVP_FAU = bVP[:, :, fau_slice[0]:fau_slice[1]]
                bVP_Bod = bVP[:, :, body_slice[0]:body_slice[1]]
                
                logits = model(bA, bVS_FAU, bVS_Bod, bVP_FAU, bVP_Bod)
                    
                # Hard-label cross entropy for validation monitoring
                y_onehot = torch.zeros_like(logits).scatter_(1, by.unsqueeze(1), 1.0)
                log_probs = F.log_softmax(logits, dim=-1)
                val_loss = -(y_onehot * (log_probs * weights.unsqueeze(0))).sum(dim=-1).mean()
                
                run_v_loss += val_loss.item()
                preds.extend(torch.max(logits, 1)[1].cpu().numpy())
                targets.extend(by.cpu().numpy())
                
        v_loss.append(run_v_loss / len(test_loader))
        v_acc.append(accuracy_score(targets, preds))
        
        # ─── EARLY STOPPING ────────────────────────────────
        if v_loss[-1] < best_loss:
            best_loss = v_loss[-1]
            best_wts = copy.deepcopy(model.state_dict())
            best_ep = epoch + 1
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= 10:
                break

    # ---- 4. Final Evaluation with Best Weights ----
    model.load_state_dict(best_wts)
    model.eval()
    f_preds, f_targets = [], []
    
    with torch.no_grad():
        for batch in test_loader:
            bA, bVS, bVP, by = [t.to(device) for t in batch]
            
            bVS_FAU = bVS[:, :, fau_slice[0]:fau_slice[1]]
            bVS_Bod = bVS[:, :, body_slice[0]:body_slice[1]]
            bVP_FAU = bVP[:, :, fau_slice[0]:fau_slice[1]]
            bVP_Bod = bVP[:, :, body_slice[0]:body_slice[1]]
            
            logits = model(bA, bVS_FAU, bVS_Bod, bVP_FAU, bVP_Bod)
            
            f_preds.extend(torch.max(logits, 1)[1].cpu().numpy())
            f_targets.extend(by.cpu().numpy())
            
    acc = accuracy_score(f_targets, f_preds)
    f1 = f1_score(f_targets, f_preds, average="macro")
    save_training_plots(exp_name, t_loss, v_loss, v_acc, best_ep, f_preds, f_targets)
    return acc, f1, best_wts

## Ablation Runner

Trains `Audio_DyadicFAUBody_MulT` across all seeds, saves the single best `.pt`, and writes results to `fusion_Audio_DyadicFAUBody.json`.

In [ ]:
# ── Load embeddings & metadata ───────────────────────────────────────────
with open(EMBEDDINGS["video_feat_dims"]) as f:
    _meta      = json.load(f)
    fau_slice  = _meta["fau_slice"]
    body_slice = _meta["body_slice"]

df = pd.read_csv(PATHS["csv"])
print("Loading embeddings...")
audio_dict        = load_embedding("audio")
self_video_dict   = load_embedding("video_self")
partner_video_dict = load_embedding("video_partner")

train_df, test_df = group_split(df, test_size=0.20, seed=42)

train_loader = DataLoader(
    DyadicAudioVisualDataset(train_df, audio_dict, self_video_dict, partner_video_dict),
    batch_size=64, shuffle=True,  num_workers=8, pin_memory=True)
test_loader = DataLoader(
    DyadicAudioVisualDataset(test_df,  audio_dict, self_video_dict, partner_video_dict),
    batch_size=64, shuffle=False, num_workers=8, pin_memory=True)

device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
A_DIM   = next(iter(audio_dict.values())).shape[-1]
FAU_DIM = fau_slice[1]  - fau_slice[0]
BOD_DIM = body_slice[1] - body_slice[0]

exp_name = "Audio_DyadicFAUBody"
acc_runs, f1_runs, best_f1, best_state = [], [], -1.0, None

for seed in RANDOM_SEEDS:
    print(f"\n{'#'*65}")
    print(f"  Audio + Speaker/Partner FAU + Body — SEED {seed}")
    print(f"{'#'*65}")
    set_seed(seed)
    model = Audio_DyadicFAUBody_MulT(audio_dim=A_DIM, fau_dim=FAU_DIM, body_dim=BOD_DIM)
    acc, f1, wts = train_eval_fusion_5stream(
        model, train_loader, test_loader, device, train_df,
        fau_slice, body_slice, f"{exp_name}_s{seed}"
    )
    acc_runs.append(acc); f1_runs.append(f1)
    if f1 > best_f1: best_f1, best_state = f1, wts

# ── Save best weights ────────────────────────────────────────────────────
save_path = os.path.join(PATHS["models"], f"{exp_name}_best.pt")
torch.save(best_state, save_path)
print(f"\nSaved best weights (F1={best_f1:.4f}) -> {save_path}")

# ── Aggregate & persist results ───────────────────────────────────────────
result_entry = {exp_name: aggregate_seeds(acc_runs, f1_runs)}
json_path = os.path.join(PATHS["results"], f"fusion_{exp_name}.json")
with open(json_path, "w") as f:
    json.dump(result_entry, f, indent=2)
print(f"Results saved → {json_path}")

r = result_entry[exp_name]
print(f"\n{exp_name}: "
      f"Acc = {r['acc_mean']*100:.2f} ± {r['acc_std']*100:.2f}%  |  "
      f"F1 = {r['f1_mean']:.4f} ± {r['f1_std']:.4f}")
print(f"(Seeds: {RANDOM_SEEDS})")